In [1]:
import heapq
import random
from bisect import bisect_left
from collections import Counter, defaultdict, deque

# 1) Binary Tree DFS

**The gist.** A tree question is a recursion question. Each node asks its children something, combines their answers with its own value, and reports one answer up to its parent.

> **Think of it like…** an org chart. To count everyone, each manager asks each direct report "how many people under you?", adds them up, adds one for themself, and tells their boss. Nobody needs to see the whole chart.

The one question to answer before coding: **what does a node need from its children, and what does it return to its parent?**

| Style | When | Shape |
|---|---|---|
| **Bottom-up** (post-order) | answer depends on subtrees: height, diameter, balanced | `left = f(node.left); right = f(node.right); combine` |
| **Top-down** (pre-order) | state flows from the root: path sum, max so far | `f(node, state)` passes updated state down |
| **Global best + local return** | the best path bends through a node, but only one branch can go up | return one branch, record `left + right` in a nonlocal |

The helper cell below builds trees from LeetCode's level-order lists (`None` = missing child), so tests look like the problem statements.

**Spot it:** height, depth, diameter, "balanced", "path sum", "ancestor", "subtree", "same / symmetric".

**Classic bugs:** forgetting the `None` base case · treating a node with one child as a leaf · returning the global best instead of the one-branch value · deep skewed trees hitting Python's recursion limit.

In [ ]:
class TreeNode:
    """Binary tree node."""

    def __init__(
        self,
        val=0,
        left=None,
        right=None,
    ):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    """Tree from LeetCode level-order list with None gaps."""
    if not values or values[0] is None:
        return None
    
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        for side in ("left", "right"):
            if i < len(values) and values[i] is not None:
                child = TreeNode(values[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root


def tree_to_list(root):
    """LeetCode level-order list of a tree (trailing None trimmed)."""
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node:
            out.append(node.val)
            queue.extend([node.left, node.right])
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out


def find(root, val):
    """First node with the given value (DFS)."""
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)


print(tree_to_list(build_tree([3, 9, 20, None, None, 15, 7])))

[3, 9, 20, None, None, 15, 7]


## 1.1 Bottom-up recursion

Recurse first, decide after. Write the return value in words before coding ("height of this subtree", "best downward chain", "(robbed, skipped)"). If the answer can pass *through* a node, keep it in a nonlocal `best` and return only what a parent can use.

### 1.1.1 [104. Maximum Depth of Binary Tree](https://leetcode.com/problems/maximum-depth-of-binary-tree/) — ★★★ · Easy

**Problem.** Number of nodes on the longest root-to-leaf path.  
**Example.** `[3, 9, 20, null, null, 15, 7]` → `3`  
**Spot it.** Node's answer = 1 + best child answer → bottom-up recursion.

**Ask first**
- Very deep (skewed) tree? → Python recursion may overflow; use BFS or an explicit stack.

> 💡 **Idea.** Depth = 1 + the deeper of the two subtrees. Empty tree = 0.

**Brute force:** none really — every node must be visited.

**Steps**
1. Empty node → 0.
2. Recurse on both children.
3. Return `1 + max(left, right)`.

**Cost:** Time O(n) (each node visited once) · Space O(h) (recursion depth (h = height; O(n) when skewed))

In [3]:
def max_depth(root):
    """Height of the tree in nodes."""
    # an empty tree has no nodes
    if not root:
        return 0
    # this node plus the taller subtree
    return 1 + max(max_depth(root.left), max_depth(root.right))


assert max_depth(build_tree([3, 9, 20, None, None, 15, 7])) == 3
assert max_depth(build_tree([1, None, 2])) == 2
assert max_depth(None) == 0
print(max_depth(build_tree([3, 9, 20, None, None, 15, 7])))

3


### 1.1.2 [100. Same Tree](https://leetcode.com/problems/same-tree/) — ★★★ · Easy

**Problem.** Are two trees structurally identical with equal values?  
**Example.** `[1, 2, 3]` and `[1, 2, 3]` → `True`; `[1, 2]` and `[1, null, 2]` → `False`  
**Spot it.** Compare roots, then recurse on both pairs of children.

**Ask first**
- Compare values only, or structure too? → Both.

> 💡 **Idea.** Two trees are the same if the roots match and both pairs of subtrees are the same.

**Brute force:** serialise both and compare strings — same O(n), more memory.

**Steps**
1. If either node is empty, they are equal only if both are.
2. Values must match.
3. Left subtrees and right subtrees must both be the same.

**Cost:** Time O(n) · Space O(h)

In [4]:
def is_same_tree(p, q):
    """True if trees are identical."""
    # both empty: nothing to compare, so they match
    if p is None and q is None:
        return True
    # only one is empty: one tree has a node here, the other does not
    if p is None or q is None:
        return False
    # roots match and both subtree pairs match
    return (
        p.val == q.val
        and is_same_tree(p.left, q.left)
        and is_same_tree(p.right, q.right)
    )


assert is_same_tree(build_tree([1, 2, 3]), build_tree([1, 2, 3]))
assert not is_same_tree(build_tree([1, 2]), build_tree([1, None, 2]))
print(is_same_tree(build_tree([1, 2, 1]), build_tree([1, 1, 2])))

False


### 1.1.3 [226. Invert Binary Tree](https://leetcode.com/problems/invert-binary-tree/) — ★★★ · Easy

**Problem.** Mirror the tree.  
**Example.** `[4, 2, 7, 1, 3, 6, 9]` → `[4, 7, 2, 9, 6, 3, 1]`  
**Spot it.** Swap children at every node.

**Ask first**
- In place or a new tree? → In place is fine.

> 💡 **Idea.** Swap left and right at every node.

**Brute force:** none — it's already O(n).

**Steps**
1. Empty → return it.
2. Set `left = invert(right)` and `right = invert(left)` simultaneously.
3. Return the node.

**Cost:** Time O(n) · Space O(h)

In [5]:
def invert_tree(root):
    """Mirror image of the tree."""
    if root:
        # swap the (already mirrored) subtrees
        root.left, root.right = invert_tree(root.right), invert_tree(root.left)
    return root


assert tree_to_list(invert_tree(build_tree([4, 2, 7, 1, 3, 6, 9]))) == [
    4,
    7,
    2,
    9,
    6,
    3,
    1,
]
print(tree_to_list(invert_tree(build_tree([2, 1, 3]))))

[2, 3, 1]


### 1.1.4 [101. Symmetric Tree](https://leetcode.com/problems/symmetric-tree/) — ★★ · Easy

**Problem.** Is the tree a mirror of itself?  
**Example.** `[1, 2, 2, 3, 4, 4, 3]` → `True`  
**Spot it.** Compare the left subtree with the mirrored right subtree.

**Ask first**
- Recursive or iterative? → Either; iterative uses a queue of pairs.

> 💡 **Idea.** Mirror check: left's left must match right's right, and left's right must match right's left.

**Brute force:** invert a copy and compare with Same Tree — O(n) extra.

**Steps**
1. `mirror(a, b)`: both empty → true; one empty → false.
2. Values must match.
3. Check `mirror(a.left, b.right)` and `mirror(a.right, b.left)`.
4. Start with `mirror(root.left, root.right)`.

**Cost:** Time O(n) · Space O(h)

In [6]:
def is_symmetric(root):
    """True if the tree mirrors itself."""

    def mirror(a, b):
        # a and b are two spots that should be mirror images of each other
        # case 1: both spots are empty -> they match
        if a is None and b is None:
            return True
        # case 2: only one spot is empty -> the shapes differ
        if a is None or b is None:
            return False
        # case 3: both have a node -> the values must match
        if a.val != b.val:
            return False
        # then outer pair and inner pair must both mirror
        return mirror(a.left, b.right) and mirror(a.right, b.left)

    return mirror(root.left, root.right) if root else True


assert is_symmetric(build_tree([1, 2, 2, 3, 4, 4, 3]))
assert not is_symmetric(build_tree([1, 2, 2, None, 3, None, 3]))
print(is_symmetric(build_tree([1, 2, 2, 3, 4, 4, 3])))

True


### 1.1.5 [572. Subtree of Another Tree](https://leetcode.com/problems/subtree-of-another-tree/) — ★★★ · Easy

**Problem.** Does `root` contain a subtree identical to `sub_root`?  
**Example.** `root = [3, 4, 5, 1, 2], sub_root = [4, 1, 2]` → `True`  
**Spot it.** At each node of `root`, test Same Tree.

**Ask first**
- Must the match go all the way down to the leaves? → Yes, the whole subtree must match.

> 💡 **Idea.** At every node of `root`, check "same tree as `sub`?".

**Brute force:** the O(m·n) check above is the expected answer; serialise + substring search gets O(m + n).

**Steps**
1. Empty `root` → false.
2. If Same Tree holds here, true.
3. Otherwise search the left subtree, then the right.

**Cost:** Time O(m · n) worst case · Space O(h)

In [ ]:
def is_same_tree(p, q):
    """True if trees are identical."""
    # both empty: nothing to compare, so they match
    if p is None and q is None:
        return True
    # only one is empty: one tree has a node here, the other does not
    if p is None or q is None:
        return False
    # roots match and both subtree pairs match
    return (
        p.val == q.val
        and is_same_tree(p.left, q.left)
        and is_same_tree(p.right, q.right)
    )


def is_subtree(root, sub_root):
    """True if sub_root appears as a subtree of root."""
    if not root:
        return False
    # match starting here, or somewhere below
    return (
        is_same_tree(root, sub_root)
        or is_subtree(root.left, sub_root)
        or is_subtree(root.right, sub_root)
    )


assert is_subtree(build_tree([3, 4, 5, 1, 2]), build_tree([4, 1, 2]))
assert not is_subtree(
    build_tree([3, 4, 5, 1, 2, None, None, None, None, 0]), build_tree([4, 1, 2])
)
print(is_subtree(build_tree([3, 4, 5, 1, 2]), build_tree([4, 1, 2])))

True


### 1.1.6 [543. Diameter of Binary Tree](https://leetcode.com/problems/diameter-of-binary-tree/) — ★★ · Easy

**Problem.** Longest path (in edges) between any two nodes.  
**Example.** `[1, 2, 3, 4, 5]` → `3` (the path 4 → 2 → 1 → 3)  
**Spot it.** Return height upward, update a global best with `left + right` at each node.

**Ask first**
- Measured in edges or nodes? → Edges.

> 💡 **Idea.** The longest path through a node = left height + right height. Return the height up, keep the best sum in a nonlocal.

**Brute force:** compute heights separately for every node, O(n²).

**Steps**
1. `height(node)` returns the subtree height (0 for empty).
2. At each node, update `best = max(best, left + right)`.
3. Return `1 + max(left, right)` to the parent.

**Cost:** Time O(n) · Space O(h)

In [8]:
def diameter_of_binary_tree(root):
    """Longest path length in edges."""
    best = 0

    def height(node):
        nonlocal best
        if not node:
            return 0
        left, right = height(node.left), height(node.right)
        # longest path that bends at this node
        best = max(best, left + right)
        # a parent can extend only one side
        return 1 + max(left, right)

    height(root)
    return best


assert diameter_of_binary_tree(build_tree([1, 2, 3, 4, 5])) == 3
assert diameter_of_binary_tree(build_tree([1, 2])) == 1
print(diameter_of_binary_tree(build_tree([1, 2, 3, 4, 5])))

3


### 1.1.7 [110. Balanced Binary Tree](https://leetcode.com/problems/balanced-binary-tree/) — ★★ · Easy

**Problem.** Is the tree height-balanced (subtree heights differ by ≤ 1 everywhere)?  
**Example.** `[3, 9, 20, null, null, 15, 7]` → `True`  
**Spot it.** Height recursion that also signals failure.

**Ask first**
- Height-balanced at *every* node? → Yes, not just the root.

> 💡 **Idea.** Return the height, or −1 as soon as any subtree is unbalanced — one pass.

**Brute force:** call a separate height function at every node, O(n²).

**Steps**
1. Empty → height 0.
2. Get both child heights.
3. If either is −1 or they differ by more than 1, return −1.
4. Otherwise return `1 + max`.

**Cost:** Time O(n) · Space O(h)

In [9]:
def is_balanced(root):
    """True if every node's subtrees differ in height by at most 1."""

    def height(node):
        if not node:
            return 0
        left, right = height(node.left), height(node.right)
        # -1 propagates "unbalanced somewhere below"
        if left < 0 or right < 0 or abs(left - right) > 1:
            return -1
        return 1 + max(left, right)

    return height(root) >= 0


assert is_balanced(build_tree([3, 9, 20, None, None, 15, 7]))
assert not is_balanced(build_tree([1, 2, 2, 3, 3, None, None, 4, 4]))
print(is_balanced(build_tree([3, 9, 20, None, None, 15, 7])))

True


### 1.1.8 [124. Binary Tree Maximum Path Sum](https://leetcode.com/problems/binary-tree-maximum-path-sum/) — ★★★ · Hard

**Problem.** Maximum sum of any path (any start and end node, values may be negative).  
**Example.** `[-10, 9, 20, null, null, 15, 7]` → `42` (15 + 20 + 7)  
**Spot it.** Diameter pattern with sums: return best downward chain, update global with both sides.

**Ask first**
- Can values be negative? → Yes → clip negative chains to 0, but the answer still has at least one node.

> 💡 **Idea.** Each node returns its best *downward* chain (max with 0 to drop negatives). The best path *through* it = node + left chain + right chain — record that globally.

**Brute force:** try every pair of nodes as path ends, O(n²).

**Steps**
1. `gain(node)` = best sum of a downward chain starting at `node`.
2. Clamp child gains at 0.
3. Update `best` with `val + left + right`.
4. Return `val + max(left, right)`.

**Cost:** Time O(n) · Space O(h)

**Likely follow-ups**
- **Return the actual path?** Store which children were used at the best node, then walk down both chains.
- **Path must go root to leaf?** Simpler top-down sum.

In [10]:
def max_path_sum(root):
    """Largest path sum anywhere in the tree."""
    best = float("-inf")

    def gain(node):
        nonlocal best
        if not node:
            return 0
        # a negative branch is better left out
        left, right = max(0, gain(node.left)), max(0, gain(node.right))
        # best path bending at this node
        best = max(best, node.val + left + right)
        # only one branch can continue upward
        return node.val + max(left, right)

    gain(root)
    return best


assert max_path_sum(build_tree([1, 2, 3])) == 6
assert max_path_sum(build_tree([-10, 9, 20, None, None, 15, 7])) == 42
assert max_path_sum(build_tree([-3])) == -3
print(max_path_sum(build_tree([-10, 9, 20, None, None, 15, 7])))

42


### 1.1.9 [236. Lowest Common Ancestor of a Binary Tree](https://leetcode.com/problems/lowest-common-ancestor-of-a-binary-tree/) — ★★ · Medium

**Problem.** Lowest node that has both `p` and `q` as descendants (a node is its own descendant).  
**Example.** `root = [3,5,1,6,2,0,8,null,null,7,4], p = 5, q = 1` → the node `3`  
**Spot it.** Bottom-up: return whichever target was found in each subtree.

**Ask first**
- Are both nodes guaranteed to be in the tree? → Yes here. If not, it's LC 1644 (see follow-ups).
- Is it a BST? → No — if it were, just compare values (LC 235).

> 💡 **Idea.** Search both sides. If p and q come back from different sides, this node is the LCA. If only one side returns something, pass it up.

**Brute force:** record the root-to-p and root-to-q paths, compare them, O(n) extra space.

**Steps**
1. Return the node if it is empty, `p` or `q`.
2. Recurse into both children.
3. Both sides non-empty → this node is the LCA.
4. Otherwise pass up whichever side is non-empty.

**Cost:** Time O(n) · Space O(h)

**Likely follow-ups**
- **p or q might not be in the tree (LC 1644)?** Don't stop early at p or q; traverse the whole tree, count how many you found, return the LCA only if both were found.
- **Nodes have parent pointers (LC 1650)?** Like linked-list intersection: walk up from p and q, switching to the other start when you hit the root — O(h), O(1) space.
- **LCA of many nodes (LC 1676)?** Same recursion, but check membership in a set of targets.
- **Many LCA queries on the same tree?** Binary lifting (O(log n) per query after O(n log n) prep) or Euler tour + sparse table.

In [ ]:
def lowest_common_ancestor(root, p, q):
    """LCA of p and q in a binary tree."""
    # found a target (or nothing): report it upward
    if not root or root is p or root is q:
        return root
    left = lowest_common_ancestor(root.left, p, q)
    right = lowest_common_ancestor(root.right, p, q)
    # targets on both sides: this node splits them
    return root if left and right else left or right


t = build_tree([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4])
assert lowest_common_ancestor(t, find(t, 5), find(t, 1)).val == 3
assert lowest_common_ancestor(t, find(t, 5), find(t, 4)).val == 5
print(lowest_common_ancestor(t, find(t, 6), find(t, 4)).val)

5


### 1.1.10 [337. House Robber III](https://leetcode.com/problems/house-robber-iii/) — ★★ · Medium

**Problem.** Houses form a binary tree; robbing two directly linked houses is forbidden. Max loot?  
**Example.** `[3, 2, 3, null, 3, null, 1]` → `7` (rob 3 + 3 + 1)  
**Spot it.** Tree DP: each node returns two values — (best if robbed, best if skipped).

**Ask first**
- Values non-negative? → Yes, so robbing is never worse unless a neighbour blocks it.

> 💡 **Idea.** Each node returns two numbers: best if I'm robbed, best if I'm skipped. Robbed = me + skipped(left) + skipped(right). Skipped = best of each child either way.

**Brute force:** plain recursion trying rob/skip at every node, exponential (memo makes it O(n)).

**Steps**
1. Empty → `(0, 0)`.
2. Robbed here → `val + left.skipped + right.skipped`.
3. Skipped here → `max(left) + max(right)`.
4. Answer = `max` of the root's pair.

**Cost:** Time O(n) · Space O(h)

**Likely follow-ups**
- **O(1) extra space besides recursion?** The tuple version already avoids a memo map — only the recursion stack (O(h)).
- **Return which houses to rob?** Keep the choice per node and walk down after: if robbed, skip children; otherwise follow each child's better option.
- **Neighbours are two levels apart instead?** Return a state per "distance from the last robbed node" — same tree DP, more states.

In [ ]:
def rob_tree(root):
    """Maximum loot with no two adjacent nodes robbed."""

    def dfs(node):
        """(best if this node is robbed, best if skipped)."""

        # empty node
        if not node:
            return 0, 0
        
        left, right = dfs(node.left), dfs(node.right)
        # robbing here forces both children to be skipped
        robbed = node.val + left[1] + right[1]
        # skipping here lets each child choose freely
        skipped = max(left) + max(right)
        return robbed, skipped

    return max(dfs(root))


assert rob_tree(build_tree([3, 2, 3, None, 3, None, 1])) == 7
assert rob_tree(build_tree([3, 4, 5, 1, 3, None, 1])) == 9
print(rob_tree(build_tree([3, 4, 5, 1, 3, None, 1])))

9


## 1.2 Top-down recursion and paths

The parent hands info *down* (remaining sum, number so far, max on the path, last direction). Check leaves explicitly. When collecting whole paths, append before recursing and pop after.

### 1.2.1 [113. Path Sum II](https://leetcode.com/problems/path-sum-ii/) — ★★ · Medium

**Problem.** All root-to-leaf paths summing to `target`.  
**Example.** `root = [5,4,8,11,null,13,4,7,2,null,null,5,1], target = 22` → `[[5,4,11,2], [5,8,4,5]]`  
**Spot it.** Top-down DFS with a path list and backtracking.

**Ask first**
- Root-to-leaf only? → Yes. Any downward path is LC 437.

> 💡 **Idea.** Walk down carrying the path and the remaining sum. At a leaf with remaining 0, save a *copy* of the path. Pop on the way back.

**Brute force:** list every root-to-leaf path, then check sums — same idea, more memory.

**Steps**
1. Append the node's value; subtract it from the remainder.
2. At a leaf with remainder 0, save a *copy* of the path.
3. Recurse into both children.
4. Pop the node's value before returning.

**Cost:** Time O(n · h) worst case (copying each matching path) · Space O(h) (the path and the recursion)

**Likely follow-ups**
- **Any downward path, not just root to leaf (LC 437)?** Prefix sums along the current path in a count map.
- **Why `path.copy()` / `path[:]`?** The same list keeps changing; without a copy every saved path ends up empty.
- **Path can start and end anywhere (even going up)?** For each node, combine downward sums from both children — similar to max path sum.

In [13]:
def path_sum_ii(root, target):
    """Every root-to-leaf path with the given sum."""
    out = []
    path = []

    def dfs(node, remaining):
        if not node:
            return
        # enter the node
        path.append(node.val)
        remaining -= node.val
        # matching leaf: save a snapshot of the path
        if not node.left and not node.right and remaining == 0:
            out.append(path.copy())
        dfs(node.left, remaining)
        dfs(node.right, remaining)
        # leave the node
        path.pop()

    dfs(root, target)
    return out


t = build_tree([5, 4, 8, 11, None, 13, 4, 7, 2, None, None, 5, 1])
assert path_sum_ii(t, 22) == [[5, 4, 11, 2], [5, 8, 4, 5]]
print(path_sum_ii(t, 22))

[[5, 4, 11, 2], [5, 8, 4, 5]]


### 1.2.2 [437. Path Sum III](https://leetcode.com/problems/path-sum-iii/) — ★★ · Medium

**Problem.** Count downward paths (any start, any end) with sum `target`.  
**Example.** `root = [10,5,-3,3,2,null,11,3,-2,null,1], target = 8` → `3`  
**Spot it.** Subarray Sum Equals K on every root-to-node path → prefix sums in a hash map.

**Ask first**
- Must paths go downward? → Yes (parent to child), but can start and end anywhere.

> 💡 **Idea.** Subarray Sum Equals K on a tree: keep a count map of prefix sums along the *current* root-to-node path. Add on the way down, remove on the way back up.

**Brute force:** start a DFS from every node, O(n²) on a skewed tree.

**Steps**
1. Seed `count = {0: 1}`.
2. At each node, add its value to the running prefix.
3. Paths ending here = `count[prefix − target]`.
4. Add `prefix` to the map, recurse, then remove it again.

**Cost:** Time O(n) · Space O(h) (the map holds one prefix per depth)

**Likely follow-ups**
- **Return the paths, not the count?** Keep the current path in a list; on a hit, the path is `path[i+1:]` for every stored index of `prefix − target`. Output can be O(n²).
- **Why remove the prefix when backtracking?** Otherwise a sibling branch would count paths that jump across the tree.
- **Values are all positive?** Still use prefix sums — the window trick doesn't map nicely onto tree branches.

In [ ]:
def path_sum_iii(root, target):
    """Number of downward paths summing to target."""
    # prefix sums on the current root-to-node path
    count = Counter({0: 1})

    def dfs(node, prefix):
        if not node:
            return 0
        prefix += node.val
        
        # paths ending here that start lower on the current path
        found = count[prefix - target]
        count[prefix] += 1
        found += dfs(node.left, prefix) + dfs(node.right, prefix)
        # leaving this node: its prefix is no longer on the path
        count[prefix] -= 1
        return found

    return dfs(root, 0)


assert path_sum_iii(build_tree([10, 5, -3, 3, 2, None, 11, 3, -2, None, 1]), 8) == 3
t = build_tree([5, 4, 8, 11, None, 13, 4, 7, 2, None, None, 5, 1])
assert path_sum_iii(t, 22) == 3
print(path_sum_iii(t, 22))

3


### 1.2.3 [129. Sum Root to Leaf Numbers](https://leetcode.com/problems/sum-root-to-leaf-numbers/) — ★★ · Medium

**Problem.** Each root-to-leaf path forms a number (digits); return the total.  
**Example.** `[1, 2, 3]` → `25` (the paths make 12 and 13)  
**Spot it.** Top-down: pass the number built so far.

**Ask first**
- Can the total overflow? → Guaranteed to fit in 32 bits here; Python doesn't care anyway.

> 💡 **Idea.** Pass the number built so far down: `num = num * 10 + val`. At a leaf, add `num` to the total.

**Brute force:** collect every path as a string, convert, add — same O(n), more memory.

**Steps**
1. `value = value * 10 + node.val`.
2. At a leaf, return `value`.
3. Otherwise return the sum of both children's results.

**Cost:** Time O(n) · Space O(h)

**Likely follow-ups**
- **Iterative?** Stack of `(node, number_so_far)`.
- **O(1) extra space?** Morris traversal with careful bookkeeping of how many digits to undo when returning — rarely expected; mention it.
- **Binary digits (LC 1022)?** Same with `num * 2 + bit`.

In [15]:
def sum_numbers(root):
    """Sum of numbers formed by root-to-leaf paths."""

    def dfs(node, value):
        if not node:
            return 0
        # append this digit
        value = value * 10 + node.val
        if not node.left and not node.right:
            return value
        return dfs(node.left, value) + dfs(node.right, value)

    return dfs(root, 0)


assert sum_numbers(build_tree([1, 2, 3])) == 25
assert sum_numbers(build_tree([4, 9, 0, 5, 1])) == 1026
print(sum_numbers(build_tree([4, 9, 0, 5, 1])))

1026


### 1.2.4 [1448. Count Good Nodes in Binary Tree](https://leetcode.com/problems/count-good-nodes-in-binary-tree/) — ★★ · Medium

**Problem.** A node is good if no node on the path from the root to it has a greater value. Count them.  
**Example.** `[3, 1, 4, 3, null, 1, 5]` → `4`  
**Spot it.** Top-down: pass the max seen so far.

**Ask first**
- Equal to the max counts as good? → Yes, `>=`.

> 💡 **Idea.** Pass the max seen on the path down. A node is good if `val >= max_so_far`.

**Brute force:** for each node, re-walk its path from the root, O(n·h).

**Steps**
1. Start with `path_max = root.val`.
2. Count the node if `val >= path_max`.
3. Recurse with `max(path_max, val)`.

**Cost:** Time O(n) · Space O(h)

**Likely follow-ups**
- **Iterative / very deep tree?** Stack of `(node, max_so_far)` — avoids recursion limits.
- **Count nodes that are the max of their whole subtree instead?** Bottom-up: return each subtree's max.

In [16]:
def good_nodes(root):
    """Number of nodes not smaller than any ancestor."""

    def dfs(node, path_max):
        if not node:
            return 0
        # nothing above is larger
        good = node.val >= path_max
        path_max = max(path_max, node.val)
        return good + dfs(node.left, path_max) + dfs(node.right, path_max)

    return dfs(root, root.val)


assert good_nodes(build_tree([3, 1, 4, 3, None, 1, 5])) == 4
assert good_nodes(build_tree([3, 3, None, 4, 2])) == 3
print(good_nodes(build_tree([3, 1, 4, 3, None, 1, 5])))

4


### 1.2.5 [1372. Longest ZigZag Path in a Binary Tree](https://leetcode.com/problems/longest-zigzag-path-in-a-binary-tree/) — ★ · Medium

**Problem.** Longest path that alternates left/right child moves (counted in edges).  
**Example.** `[1,null,1,1,1,null,null,1,1,null,1,null,null,null,1]` → `3`  
**Spot it.** Top-down state: (direction of last move, current length).

**Ask first**
- Length = edges? → Yes; a single node has length 0.

> 💡 **Idea.** Pass down the zigzag length and the direction you arrived from. Going the *other* way extends it (+1); going the same way restarts at 1.

**Brute force:** start a zigzag walk from every node, O(n·h).

**Steps**
1. Start at both children of the root with length 1.
2. Going left after a right move → `length + 1`; going left after a left move → 1.
3. Symmetric for right.
4. Track the maximum length.

**Cost:** Time O(n) · Space O(h)

**Likely follow-ups**
- **Return the path itself?** Track the start node when the length resets and the best end node.
- **Bottom-up version?** Return `(longest going left, longest going right)` from each node: `left = 1 + child_left.right`, etc.

In [17]:
def longest_zig_zag(root):
    """Longest alternating downward path length."""
    best = 0

    def dfs(node, went_left, length):
        nonlocal best
        if not node:
            return
        best = max(best, length)
        # going left continues the zigzag only if we just went right
        dfs(node.left, True, 1 if went_left else length + 1)
        # going right continues it only if we just went left
        dfs(node.right, False, length + 1 if went_left else 1)

    dfs(root.left, True, 1)
    dfs(root.right, False, 1)
    return best


t = build_tree([1, None, 1, 1, 1, None, None, 1, 1, None, 1, None, None, None, 1])
assert longest_zig_zag(t) == 3
assert longest_zig_zag(build_tree([1, 1, 1, None, 1, None, None, 1, 1, None, 1])) == 4
assert longest_zig_zag(build_tree([1])) == 0
print(longest_zig_zag(t))

3


## 1.3 Construction, transformation and serialisation

Preorder starts with the root, postorder ends with it, inorder puts the root between its subtrees. With a `value → inorder index` map, every subtree is just an index range.

### 1.3.1 [105. Construct Binary Tree from Preorder and Inorder Traversal](https://leetcode.com/problems/construct-binary-tree-from-preorder-and-inorder-traversal/) — ★★★ · Medium

**Problem.** Rebuild the tree from its preorder and inorder sequences (values distinct).  
**Example.** `preorder = [3, 9, 20, 15, 7], inorder = [9, 3, 15, 20, 7]` → `[3, 9, 20, null, null, 15, 7]`  
**Spot it.** Preorder gives the root; inorder splits left and right subtrees.

**Ask first**
- Unique values? → Yes — that's what makes the split possible. With duplicates the tree isn't unique.

> 💡 **Idea.** The next preorder value is the root. Find it in inorder (with a value → index map): everything left of it is the left subtree, everything right is the right subtree. Recurse.

**Brute force:** search inorder linearly for each root, O(n²).

**Steps**
1. Map each value to its inorder index.
2. `make(lo, hi)`: empty range → `None`.
3. Take the next preorder value as the root; find its inorder index `m`.
4. Build left from `(lo, m − 1)` then right from `(m + 1, hi)` — order matters, preorder is consumed left first.

**Cost:** Time O(n) · Space O(n) (the map and recursion)

**Likely follow-ups**
- **Inorder + postorder (LC 106)?** Take roots from the *end* of postorder and build the right subtree first.
- **Preorder + postorder (LC 889)?** Not unique in general; use `preorder[1]` (the left child) to find the left subtree's size in postorder.
- **Duplicate values?** Can't decide the split uniquely — ask the interviewer, or return any valid tree.
- **Iterative version?** Stack + inorder pointer: push nodes as left children until the top equals `inorder[i]`, then pop and attach the next as a right child. O(n).

In [18]:
def build_from_pre_in(preorder, inorder):
    """Tree with the given preorder and inorder traversals."""
    if not preorder:
        return None

    # preorder: the first value is always the root
    root = TreeNode(preorder[0])
    # inorder: everything left of the root is the left subtree
    m = inorder.index(root.val)

    root.left = build_from_pre_in(preorder[1:m + 1], inorder[:m])
    root.right = build_from_pre_in(preorder[m + 1:], inorder[m + 1:])
    return root


t = build_from_pre_in([3, 9, 20, 15, 7], [9, 3, 15, 20, 7])
assert tree_to_list(t) == [3, 9, 20, None, None, 15, 7]
print(tree_to_list(t))

[3, 9, 20, None, None, 15, 7]


### 1.3.2 [106. Construct Binary Tree from Inorder and Postorder Traversal](https://leetcode.com/problems/construct-binary-tree-from-inorder-and-postorder-traversal/) — ★★ · Medium

**Problem.** Rebuild the tree from inorder and postorder sequences.  
**Example.** `inorder = [9, 3, 15, 20, 7], postorder = [9, 15, 7, 20, 3]` → `[3, 9, 20, null, null, 15, 7]`  
**Spot it.** Postorder read backwards is root → right → left.

**Ask first**
- Unique values? → Yes, otherwise the split is ambiguous.

> 💡 **Idea.** Postorder ends with the root. Take roots from the back, split inorder at the root, and build the **right** subtree first (that's the order you meet them going backwards).

**Brute force:** linear search in inorder for each root, O(n²).

**Steps**
1. Map values to inorder indices.
2. Read postorder backwards.
3. Each value is the root of the current range.
4. Build the right range first, then the left.

**Cost:** Time O(n) · Space O(n)

**Likely follow-ups**
- **Why build the right subtree first?** Reading postorder backwards gives root, right, left.
- **Iterative?** Mirror of the 105 stack method, scanning postorder and inorder from the end.

In [19]:
def build_from_in_post(inorder, postorder):
    """Tree with the given inorder and postorder traversals."""
    if not postorder:
        return None

    # postorder: the last value is always the root
    root = TreeNode(postorder[-1])
    # inorder: everything left of the root is the left subtree
    m = inorder.index(root.val)

    root.left = build_from_in_post(inorder[:m], postorder[:m])
    root.right = build_from_in_post(inorder[m + 1:], postorder[m:-1])
    return root


t = build_from_in_post([9, 3, 15, 20, 7], [9, 15, 7, 20, 3])
assert tree_to_list(t) == [3, 9, 20, None, None, 15, 7]
print(tree_to_list(t))

[3, 9, 20, None, None, 15, 7]


### 1.3.3 [114. Flatten Binary Tree to Linked List](https://leetcode.com/problems/flatten-binary-tree-to-linked-list/) — ★★ · Medium

**Problem.** Flatten in place into a right-skewed "list" in preorder.  
**Example.** `[1, 2, 5, 3, 4, null, 6]` → `1→2→3→4→5→6`, all through the `right` pointers  
**Spot it.** Morris-style: splice each left subtree between the node and its right subtree.

**Ask first**
- Order? → Preorder, using `right` as next and `left = None`.
- In place? → Yes.

> 💡 **Idea.** For each node with a left child: find the rightmost node of the left subtree, hang the right subtree there, move the left subtree to the right, clear left. Then move right.

**Brute force:** collect nodes in preorder into a list, then relink — O(n) extra space.

**Steps**
1. At `cur` with a left child, find the rightmost node of the left subtree.
2. Attach `cur.right` to that node.
3. Move the left subtree to `cur.right`; clear `cur.left`.
4. Step to `cur.right` and repeat.

**Cost:** Time O(n) (each node is on a "rightmost" walk at most once) · Space O(1)

**Likely follow-ups**
- **O(1) extra space?** The rightmost-of-left trick above (Morris-style) — no stack, no recursion.
- **Recursive version?** Reverse preorder (right, left, node) with a `prev` pointer: `node.right = prev; node.left = None; prev = node`.
- **Flatten into a doubly linked list in *inorder* (LC 426)?** Inorder walk linking `prev.right = node; node.left = prev`, then close the ring.

In [20]:
def flatten(root):
    """Flatten tree to a preorder right-chain in place."""
    cur = root
    while cur:
        if cur.left:
            # last node of the left subtree in preorder
            tail = cur.left
            while tail.right:
                tail = tail.right
            # right subtree follows it
            tail.right = cur.right
            # left subtree moves to the right
            cur.right, cur.left = cur.left, None
        cur = cur.right


t = build_tree([1, 2, 5, 3, 4, None, 6])
flatten(t)
assert tree_to_list(t) == [1, None, 2, None, 3, None, 4, None, 5, None, 6]
print(tree_to_list(t))

[1, None, 2, None, 3, None, 4, None, 5, None, 6]


### 1.3.4 [297. Serialize and Deserialize Binary Tree](https://leetcode.com/problems/serialize-and-deserialize-binary-tree/) — ★★★ · Hard

**Problem.** Encode a tree to a string and decode it back.  
**Example.** `[1, 2, 3, null, null, 4, 5]` → turn into a string → turn back into the same tree  
**Spot it.** Preorder with explicit null markers is unambiguous.

**Ask first**
- Any format allowed? → Yes, as long as it round-trips.

> 💡 **Idea.** Preorder with `#` for empty children uniquely describes a tree. Deserialise by reading tokens in the same preorder.

**Brute force:** store level order with nulls — also fine, just different.

**Steps**
1. Serialise: preorder, writing `#` for `None`, joined by commas.
2. Deserialise: an iterator over tokens.
3. `#` → `None`; otherwise create the node and build its left, then right subtree.

**Cost:** Time O(n) · Space O(n)

**Likely follow-ups**
- **Make it compact?** For a BST (LC 449), preorder alone is enough (rebuild using bounds) — no null markers.
- **N-ary tree (LC 428)?** Write each node's value followed by its child count.

In [21]:
def serialize(root):
    """Preorder string with # for null."""
    out = []

    def dfs(node):
        # explicit marker for every missing child
        if not node:
            out.append("#")
            return
        out.append(str(node.val))
        dfs(node.left)
        dfs(node.right)

    dfs(root)
    return ",".join(out)


def deserialize(data):
    """Inverse of serialize."""
    tokens = iter(data.split(","))

    def make():
        # read tokens in the same preorder they were written
        tok = next(tokens)
        if tok == "#":
            return None
        return TreeNode(int(tok), make(), make())

    return make()


t = build_tree([1, 2, 3, None, None, 4, 5])
assert tree_to_list(deserialize(serialize(t))) == [1, 2, 3, None, None, 4, 5]
assert deserialize(serialize(None)) is None
print(serialize(t))

1,2,#,#,3,4,#,#,5,#,#


### 1.3.5 [427. Construct Quad Tree](https://leetcode.com/problems/construct-quad-tree/) — ★★ · Medium

**Problem.** Build a quad tree for an `n × n` grid of 0s and 1s: a square that holds one value is a leaf; otherwise it splits into four quarters.  
**Example.** `[[1, 1], [1, 1]]` → one leaf with value 1; `[[1, 0], [0, 1]]` → an internal node with four leaves  
**Spot it.** "Split into four quarters until uniform" → divide and conquer on squares.

**Ask first**
- Is n a power of two? → Yes, so quadrants always split evenly.

> 💡 **Idea.** If a square is all one value, it's a leaf. Otherwise split into four quadrants and recurse. (Build bottom-up and merge four equal leaves to avoid rescanning.)

**Brute force:** re-scan every square at every level, O(n² log n).

**Steps**
1. A 1 × 1 square is a leaf holding that cell's value.
2. Otherwise build the four quarters: top-left, top-right, bottom-left, bottom-right.
3. Four leaves with one shared value collapse into one leaf; anything else becomes an internal node.

**Cost:** Time O(n²) (each cell is read once, each merge is O(1)) · Space O(log n) recursion depth, plus the nodes of the tree

**Likely follow-ups**
- **O(n²) instead of O(n² log n)?** Build bottom-up: recurse first, merge if all four children are leaves with the same value — or use a 2D prefix sum to test "all same" in O(1).
- **OR of two quad trees (LC 558)?** If either is a leaf with 1 → leaf 1; if a leaf with 0 → return the other; else recurse on the four pairs and merge equal leaves.
- **Why use a quad tree at all?** Images / maps with big uniform areas compress well; also used for spatial lookups.

In [22]:
class QuadNode:
    """Quad-tree node; a leaf has no children."""

    def __init__(self, val, is_leaf, children=None):
        self.val = val
        self.is_leaf = is_leaf
        # top-left, top-right, bottom-left, bottom-right
        self.children = children or []


def construct_quad_tree(grid):
    """Quad tree of a 2^k × 2^k grid of 0s and 1s."""

    def build(row, col, size):
        # a single cell is always uniform
        if size == 1:
            return QuadNode(grid[row][col], True)
        half = size // 2
        # build the four quarters first
        kids = [
            build(row, col, half),
            build(row, col + half, half),
            build(row + half, col, half),
            build(row + half, col + half, half),
        ]
        # four leaves with one value collapse into a single leaf
        if all(kid.is_leaf for kid in kids) and len({kid.val for kid in kids}) == 1:
            return QuadNode(kids[0].val, True)
        return QuadNode(1, False, kids)

    return build(0, 0, len(grid))


def quad_to_list(node):
    """A leaf's value, or the four children as a nested list."""
    if node.is_leaf:
        return node.val
    return [quad_to_list(kid) for kid in node.children]


assert quad_to_list(construct_quad_tree([[1, 1], [1, 1]])) == 1
assert quad_to_list(construct_quad_tree([[1, 0], [0, 1]])) == [1, 0, 0, 1]
grid = [[1, 1, 0, 0], [1, 1, 0, 0], [1, 1, 1, 1], [1, 1, 1, 1]]
assert quad_to_list(construct_quad_tree(grid)) == [1, 0, 1, 1]
print(quad_to_list(construct_quad_tree(grid)))

[1, 0, 1, 1]


### 1.3.6 [863. All Nodes Distance K in Binary Tree](https://leetcode.com/problems/all-nodes-distance-k-in-binary-tree/) — ★★ · Medium

**Problem.** Values of all nodes at distance `k` from a `target` node.  
**Example.** `root = [3,5,1,6,2,0,8,null,null,7,4], target = 5, k = 2` → `[7, 4, 1]`  
**Spot it.** Distance can go upward → turn the tree into an undirected graph, then BFS.

**Ask first**
- Unique values? → Yes, so a `seen` set of nodes (or values) works.

> 💡 **Idea.** A tree is a graph you can only walk down. Add parent links (a dict), then BFS k steps out from the target in all three directions (left, right, parent).

**Brute force:** for every node, compute its distance to target via LCA, O(n²).

**Steps**
1. DFS to record each node's parent.
2. BFS from `target` over left, right and parent edges.
3. Keep a visited set (the graph now has back edges).
4. After `k` levels, the frontier is the answer.

**Cost:** Time O(n) · Space O(n)

**Likely follow-ups**
- **Without a parent map?** One DFS that returns the distance from the target to this node; when an ancestor is d away, collect nodes at depth `k − d − 1` in its *other* subtree.
- **Distance between two nodes?** `depth(p) + depth(q) − 2·depth(LCA)`.
- **Many queries with different targets/k?** Build the adjacency list once and BFS per query.

In [23]:
def distance_k(root, target, k):
    """Values at exactly k edges from target."""
    # parent pointers turn the tree into an undirected graph
    parent = {root: None}
    stack = [root]
    while stack:
        node = stack.pop()
        for child in (node.left, node.right):
            if child:
                parent[child] = node
                stack.append(child)
    # BFS k levels out from target
    frontier, seen = [target], {target}
    for _ in range(k):
        nxt = []
        for node in frontier:
            for nb in (node.left, node.right, parent[node]):
                if nb and nb not in seen:
                    seen.add(nb)
                    nxt.append(nb)
        frontier = nxt
    return sorted(node.val for node in frontier)


t = build_tree([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4])
assert distance_k(t, find(t, 5), 2) == [1, 4, 7]
single = build_tree([1])
assert distance_k(single, single, 3) == []
print(distance_k(t, find(t, 5), 2))

[1, 4, 7]


---
# 2) Binary Tree BFS

**The gist.** Visit the tree one level at a time with a queue. Snapshot the queue size at the start of a level — exactly that many nodes belong to it.

> **Think of it like…** ripples on a pond: everything one step from the stone gets touched first, then two steps, and so on.

```python
queue = deque([root])
while queue:
    level = []
    for _ in range(len(queue)):        # exactly the nodes of the current level
        node = queue.popleft()
        level.append(node.val)
        queue.extend(c for c in (node.left, node.right) if c)
```

**Spot it:** "level by level", "right side view", "zigzag", "per level max / sum / average", "width", "minimum depth", "vertical order".

**Classic bugs:** using `len(queue)` inside the loop instead of a snapshot · pushing `None` children · forgetting the empty-tree case.

## 2.1 Level-order traversal

Every question here is the level template plus one twist: what you record per level, what rides along with each node (an index, a column), or how you handle gaps.

### 2.1.1 [102. Binary Tree Level Order Traversal](https://leetcode.com/problems/binary-tree-level-order-traversal/) — ★★★ · Medium

**Problem.** Values level by level, left to right.  
**Example.** `[3, 9, 20, null, null, 15, 7]` → `[[3], [9, 20], [15, 7]]`  
**Spot it.** Level by level → BFS with the level-size loop.

**Ask first**
- Return values grouped per level? → Yes, a list of lists.

> 💡 **Idea.** BFS with a queue; snapshot `len(queue)` at the start of each level and pop exactly that many.

**Brute force:** DFS with a depth parameter appending to `levels[depth]` — also fine, O(n).

**Steps**
1. Queue the root.
2. Snapshot `len(queue)`; pop that many nodes, collecting their values.
3. Push each popped node's children.
4. Append the level and repeat.

**Cost:** Time O(n) · Space O(width) (the queue holds at most one or two levels)

**Likely follow-ups**
- **Bottom-up order (LC 107)?** Same BFS, reverse the result (or prepend each level).
- **DFS instead of BFS?** Pass `depth`; create `levels[depth]` the first time you reach that depth; left before right keeps order.
- **N-ary tree?** Extend the queue with `node.children`.

In [24]:
def level_order(root):
    """Node values grouped by level."""
    out = []
    queue = deque([root] if root else [])
    while queue:
        level = []
        # the queue holds exactly this level right now
        for _ in range(len(queue)):
            node = queue.popleft()
            level.append(node.val)
            # children form the next level
            queue.extend(c for c in (node.left, node.right) if c)
        out.append(level)
    return out


assert level_order(build_tree([3, 9, 20, None, None, 15, 7])) == [[3], [9, 20], [15, 7]]
assert level_order(None) == []
print(level_order(build_tree([3, 9, 20, None, None, 15, 7])))

[[3], [9, 20], [15, 7]]


### 2.1.2 [199. Binary Tree Right Side View](https://leetcode.com/problems/binary-tree-right-side-view/) — ★★ · Medium

**Problem.** Values visible from the right side (last node of each level).  
**Example.** `[1, 2, 3, null, 5, null, 4]` → `[1, 3, 4]`  
**Spot it.** Level-order, keep the last value of each level.

**Ask first**
- Top to bottom? → Yes, one value per level.

> 💡 **Idea.** BFS by level and keep the last node of each level. (Or DFS right-first and record the first node seen at each depth.)

**Brute force:** collect full levels, then take each last element — same O(n).

**Steps**
1. Run level order.
2. Take the last value of every level.

**Cost:** Time O(n) · Space O(n) here (the levels); O(width) if recorded on the fly

**Likely follow-ups**
- **Left side view?** Keep the first node of each level (or DFS left-first).
- **Both views / the outline (boundary of a tree, LC 545)?** Left boundary + leaves + reversed right boundary.
- **Top view / bottom view?** Track a column index; keep the first (top) or last (bottom) node per column in BFS order.

In [25]:
def right_side_view(root):
    """Rightmost value on each level."""
    # the last node of a level is the one visible from the right
    return [level[-1] for level in level_order(root)]


assert right_side_view(build_tree([1, 2, 3, None, 5, None, 4])) == [1, 3, 4]
assert right_side_view(build_tree([1, 2, 3, 4, None, None, None, 5])) == [1, 3, 4, 5]
print(right_side_view(build_tree([1, 2, 3, None, 5, None, 4])))

[1, 3, 4]


### 2.1.3 [103. Binary Tree Zigzag Level Order Traversal](https://leetcode.com/problems/binary-tree-zigzag-level-order-traversal/) — ★★ · Medium

**Problem.** Level order, alternating left→right and right→left.  
**Example.** `[3, 9, 20, null, null, 15, 7]` → `[[3], [20, 9], [15, 7]]`  
**Spot it.** Level BFS, reverse every other level.

**Ask first**
- First level left to right? → Yes.

> 💡 **Idea.** Normal level-order BFS; reverse every other level (or append to the left of a deque on odd levels).

**Brute force:** level order, then reverse odd levels — that *is* the simple solution.

**Steps**
1. Run level order.
2. Reverse levels with odd index.

**Cost:** Time O(n) · Space O(n)

**Likely follow-ups**
- **Without reversing lists?** Two stacks: pop from the current one, push children left-then-right on one level and right-then-left on the next.
- **With a deque only?** Even levels: pop left, push children right. Odd levels: pop right, push children (right, then left) on the left.

In [26]:
def zigzag_level_order(root):
    """Level order with alternating direction."""
    levels = level_order(root)
    # odd levels read right to left
    return [lvl if i % 2 == 0 else lvl[::-1] for i, lvl in enumerate(levels)]


assert zigzag_level_order(build_tree([3, 9, 20, None, None, 15, 7])) == [
    [3],
    [20, 9],
    [15, 7],
]
print(zigzag_level_order(build_tree([1, 2, 3, 4, None, None, 5])))

[[1], [3, 2], [4, 5]]


### 2.1.4 [1161. Maximum Level Sum of a Binary Tree](https://leetcode.com/problems/maximum-level-sum-of-a-binary-tree/) — ★ · Medium

**Problem.** The smallest level number (root = 1) with the maximum sum.  
**Example.** `[1, 7, 0, 7, -8, null, null]` → `2` (level 2 has sum 7)  
**Spot it.** Level BFS.

**Ask first**
- Ties? → Return the smallest level (use `>`, not `>=`).
- 1-indexed levels? → Yes.

> 💡 **Idea.** BFS level by level, sum each level, remember the first level with the biggest sum.

**Brute force:** DFS into a list of level sums — also O(n).

**Steps**
1. Sum each level.
2. Return the 1-based index of the first maximum.

**Cost:** Time O(n) · Space O(n)

**Likely follow-ups**
- **DFS version?** `sums[depth] += val`; then `argmax`. Uses O(height) stack instead of O(width) queue.
- **Max level *average* (LC 637)?** Divide by the level size.

In [27]:
def max_level_sum(root):
    """1-based level with the largest sum."""
    sums = [sum(lvl) for lvl in level_order(root)]
    # index() picks the smallest level on ties
    return sums.index(max(sums)) + 1


assert max_level_sum(build_tree([1, 7, 0, 7, -8, None, None])) == 2
assert (
    max_level_sum(
        build_tree([989, None, 10250, 98693, -89388, None, None, None, -32127])
    )
    == 2
)
print(max_level_sum(build_tree([1, 7, 0, 7, -8, None, None])))

2


### 2.1.5 [662. Maximum Width of Binary Tree](https://leetcode.com/problems/maximum-width-of-binary-tree/) — ★★ · Medium

**Problem.** Maximum width of any level, counting null gaps between the end nodes.  
**Example.** `[1, 3, 2, 5, 3, null, 9]` → `4`  
**Spot it.** Give nodes heap-style indices (`2i`, `2i + 1`); width = last − first + 1.

**Ask first**
- Do gaps between nodes count? → Yes, that's why you need positions, not counts.

> 💡 **Idea.** Number positions like a heap: left child `2i`, right child `2i + 1`. Width of a level = last index − first index + 1.

**Brute force:** BFS with `None` placeholders — exponential on a skewed tree.

**Steps**
1. Queue `(root, 0)`.
2. For each level, note the first index and re-base indices by subtracting it.
3. Children get `2i` and `2i + 1`.
4. Width = last re-based index + 1; keep the max.

**Cost:** Time O(n) · Space O(width)

**Likely follow-ups**
- **Indices overflow on deep trees?** Subtract the level's first index from every index on that level (keeps numbers small, widths unchanged).
- **DFS version?** Store the first index seen at each depth; width = `i − first[depth] + 1`.

In [28]:
def width_of_binary_tree(root):
    """Largest level width including gaps."""
    best = 0
    queue = deque([(root, 0)])
    while queue:
        first = queue[0][1]
        last = first
        for _ in range(len(queue)):
            node, idx = queue.popleft()
            # re-base so indices stay small
            idx -= first
            last = idx
            # heap numbering for children
            if node.left:
                queue.append((node.left, 2 * idx))
            if node.right:
                queue.append((node.right, 2 * idx + 1))
        best = max(best, last + 1)
    return best


assert width_of_binary_tree(build_tree([1, 3, 2, 5, 3, None, 9])) == 4
assert width_of_binary_tree(build_tree([1, 3, 2, 5, None, None, 9, 6, None, 7])) == 7
assert width_of_binary_tree(build_tree([1, 3, 2, 5])) == 2
print(width_of_binary_tree(build_tree([1, 3, 2, 5, 3, None, 9])))

4


### 2.1.6 [117. Populating Next Right Pointers in Each Node II](https://leetcode.com/problems/populating-next-right-pointers-in-each-node-ii/) — ★★ · Medium

**Problem.** Set each node's `next` to its right neighbour on the same level, O(1) extra space.  
**Example.** `[1, 2, 3, 4, 5, null, 7]` → `2`.next = `3`, `4`.next = `5`, `5`.next = `7`  
**Spot it.** Use the already-linked level above as a linked list to wire the level below.

**Ask first**
- Perfect tree? → No (that's LC 116) — children can be missing, so use a dummy + tail pointer.
- O(1) extra space? → Yes — so no queue.

> 💡 **Idea.** Use the `next` links you already built on one level to walk across it, while linking up the children for the level below (a dummy head collects them).

**Brute force:** BFS with a queue linking neighbours, O(width) space.

**Steps**
1. `level` starts at the root.
2. Walk the level via `next`, appending each child to a new chain (dummy + tail).
3. End the chain with `None`.
4. Move `level` to `dummy.next` and repeat.

**Cost:** Time O(n) · Space O(1)

**Likely follow-ups**
- **Why does the queue break the rule?** It holds a whole level — O(n) for wide trees.
- **Perfect binary tree (LC 116)?** Simpler: `node.left.next = node.right; node.right.next = node.next.left`.

In [29]:
def connect(root):
    """Link each node to its right neighbour."""
    level = root
    while level:
        # build the child level as a chain
        dummy = tail = TreeNode()
        node = level
        # walk the current level through its next links
        while node:
            for child in (node.left, node.right):
                if child:
                    tail.next = tail = child
            node = getattr(node, "next", None)
        tail.next = None
        # descend to the first node of the child level
        level = dummy.next
    return root


t = connect(build_tree([1, 2, 3, 4, 5, None, 7]))
row, node = [], t.left.left
while node:
    row.append(node.val)
    node = node.next
assert row == [4, 5, 7] and t.left.next is t.right
print(row)

[4, 5, 7]


### 2.1.7 [314. Binary Tree Vertical Order Traversal](https://leetcode.com/problems/binary-tree-vertical-order-traversal/) — ★ · Medium · Premium

**Problem.** Group values by column (root at 0, left −1, right +1), top to bottom, left to right.  
**Example.** `[3, 9, 20, null, null, 15, 7]` → `[[9], [3, 15], [20], [7]]`  
**Spot it.** BFS (for top-to-bottom order) carrying the column index.

**Ask first**
- Same row and column — which first? → Left to right (BFS order). LC 987 instead sorts ties by value.

> 💡 **Idea.** BFS carrying a column index (left `c − 1`, right `c + 1`). Append each value to `cols[c]`; BFS order keeps top-to-bottom and left-to-right.

**Brute force:** DFS + sort by `(col, row)` — also works, O(n log n).

**Steps**
1. BFS with `(node, col)`.
2. Append each value to `columns[col]`.
3. Left child gets `col − 1`, right child `col + 1`.
4. Output columns from the smallest key to the largest.

**Cost:** Time O(n) (+ O(w log w) to sort w column keys) · Space O(n)

**Likely follow-ups**
- **Ties sorted by value (LC 987)?** Collect `(col, row, val)` and sort all of them. O(n log n).
- **Avoid sorting columns?** Track `min_col` and `max_col` and read `cols[min..max]`.
- **DFS instead of BFS?** Need the row too, then sort by `(col, row)`; otherwise deeper-left nodes can come out first.

In [30]:
def vertical_order(root):
    """Columns of values, left to right."""
    if not root:
        return []
    columns = defaultdict(list)
    queue = deque([(root, 0)])
    while queue:
        node, col = queue.popleft()
        # BFS order = top-to-bottom, left-to-right within a column
        columns[col].append(node.val)
        if node.left:
            queue.append((node.left, col - 1))
        if node.right:
            queue.append((node.right, col + 1))
    return [columns[c] for c in sorted(columns)]


assert vertical_order(build_tree([3, 9, 20, None, None, 15, 7])) == [
    [9],
    [3, 15],
    [20],
    [7],
]
assert vertical_order(build_tree([3, 9, 8, 4, 0, 1, 7])) == [
    [4],
    [9],
    [3, 0, 1],
    [8],
    [7],
]
print(vertical_order(build_tree([3, 9, 20, None, None, 15, 7])))

[[9], [3, 15], [20], [7]]


---
# 3) Binary Search Tree

**The gist.** Smaller keys go left, bigger keys go right — at every node. So you find things by going one way at each step, and an in-order walk reads the keys in sorted order.

> **Think of it like…** a phone book you can only open in the middle: each look tells you which half to keep.

| Fact | Use it for |
|---|---|
| **In-order is sorted** | k-th smallest, validate, iterators, min difference |
| **Each lookup goes one way** (O(h)) | search, insert, delete, LCA, successor |

**Spot it:** "BST", "sorted order", "k-th smallest", "successor / predecessor", "closest value".

**Classic bugs:** validating only parent vs child (the rule covers the whole subtree — pass `(low, high)` bounds) · forgetting O(h) is O(n) on a skewed tree · not asking how duplicates are handled.

## 3.1 BST properties and in-order traversal

Pick one of two tools: **descend** (compare and go one way, O(h)) or **in-order** (visit in sorted order, often with a `prev` pointer).

### 3.1.1 [98. Validate Binary Search Tree](https://leetcode.com/problems/validate-binary-search-tree/) — ★★★ · Medium

**Problem.** Is the tree a valid BST (strict inequalities)?  
**Example.** `[2, 1, 3]` → `True`; `[5, 1, 4, null, null, 3, 6]` → `False`  
**Spot it.** Each node must lie in an open interval inherited from its ancestors.

**Ask first**
- Duplicates allowed? → No, strictly less/greater. If duplicates go right, change one side to `<=`.

> 💡 **Idea.** Pass an allowed range `(low, high)` down. Going left tightens `high`; going right tightens `low`. Every node must sit strictly inside its range.

**Brute force:** check only parent vs child — wrong (a deep node can break an ancestor's rule).

**Steps**
1. Start with `(−∞, +∞)`.
2. A node outside `(low, high)` → invalid.
3. Left child gets `(low, node.val)`; right child gets `(node.val, high)`.
4. Empty subtrees are valid.

**Cost:** Time O(n) · Space O(h)

**Likely follow-ups**
- **Iterative?** In-order with a stack; it must be strictly increasing (compare to `prev`).
- **O(1) space?** Morris in-order traversal while tracking `prev`.
- **Recover two swapped nodes (LC 99)?** In-order; find the first and last "drops", swap those values.

In [31]:
def is_valid_bst(root):
    """True if the tree satisfies the BST property."""

    def valid(node, low, high):
        if not node:
            return True
        # must respect every ancestor, summarised by (low, high)
        if not low < node.val < high:
            return False
        # left side caps at node.val, right side floors at it
        return valid(node.left, low, node.val) and valid(node.right, node.val, high)

    return valid(root, float("-inf"), float("inf"))


assert is_valid_bst(build_tree([2, 1, 3]))
assert not is_valid_bst(build_tree([5, 1, 4, None, None, 3, 6]))
assert not is_valid_bst(build_tree([5, 4, 6, None, None, 3, 7]))
print(is_valid_bst(build_tree([2, 1, 3])))

True


### 3.1.2 [230. Kth Smallest Element in a BST](https://leetcode.com/problems/kth-smallest-element-in-a-bst/) — ★★★ · Medium

**Problem.** k-th smallest key in a BST.  
**Example.** `root = [3, 1, 4, null, 2], k = 1` → `1`  
**Spot it.** In-order = sorted → stop at the k-th visit.

**Ask first**
- Will the tree change between queries? → Not here; if it does, see the follow-up.

> 💡 **Idea.** In-order walk is sorted — stop at the k-th node.

**Brute force:** collect the whole in-order list and index it — O(n) space.

**Steps**
1. Push the left spine onto a stack.
2. Pop a node: it is the next smallest; decrement `k`.
3. If `k` hits 0, return it.
4. Otherwise move to its right child and push that left spine.

**Cost:** Time O(h + k) · Space O(h)

**Likely follow-ups**
- **Tree changes often and you query k-th a lot?** Store subtree sizes in each node; walk down comparing k with the left size. O(h) per query, update sizes on insert/delete.
- **k-th largest?** Reverse in-order (right, node, left).
- **Without recursion?** Iterative in-order with a stack, stopping after k pops — O(h + k).

In [32]:
def kth_smallest(root, k):
    """k-th smallest value in a BST."""
    stack = []
    node = root
    while True:
        # go as far left as possible
        while node:
            stack.append(node)
            node = node.left
        # next node in sorted order
        node = stack.pop()
        k -= 1
        if k == 0:
            return node.val
        # continue with the larger keys
        node = node.right


assert kth_smallest(build_tree([3, 1, 4, None, 2]), 1) == 1
assert kth_smallest(build_tree([5, 3, 6, 2, 4, None, None, 1]), 3) == 3
print(kth_smallest(build_tree([5, 3, 6, 2, 4, None, None, 1]), 3))

3


### 3.1.3 [235. Lowest Common Ancestor of a Binary Search Tree](https://leetcode.com/problems/lowest-common-ancestor-of-a-binary-search-tree/) — ★★★ · Medium

**Problem.** LCA of `p` and `q` in a BST.  
**Example.** `root = [6,2,8,0,4,7,9,null,null,3,5], p = 2, q = 8` → the node `6`  
**Spot it.** The LCA is the first node whose value lies between `p` and `q`.

**Ask first**
- Are both nodes in the tree? → Yes, guaranteed.

> 💡 **Idea.** If both values are smaller, go left; both bigger, go right. The first node where they split (or one equals the node) is the LCA.

**Brute force:** use the general LC 236 solution, O(n) — ignores the BST order.

**Steps**
1. Both values smaller than the node → go left.
2. Both larger → go right.
3. Otherwise return the node.

**Cost:** Time O(h) · Space O(1)

**Likely follow-ups**
- **O(1) space?** Do it iteratively — no recursion stack.
- **Not a BST?** Use LC 236.
- **What if p is an ancestor of q?** Then the walk stops at p (it's "between" itself and q) — p is the answer.

In [33]:
def lca_bst(root, p, q):
    """LCA in a BST."""
    node = root
    while True:
        # both targets lie left
        if p.val < node.val and q.val < node.val:
            node = node.left
        # both targets lie right
        elif p.val > node.val and q.val > node.val:
            node = node.right
        # they split here (or one of them is here)
        else:
            return node


t = build_tree([6, 2, 8, 0, 4, 7, 9, None, None, 3, 5])
assert lca_bst(t, find(t, 2), find(t, 8)).val == 6
assert lca_bst(t, find(t, 2), find(t, 4)).val == 2
print(lca_bst(t, find(t, 3), find(t, 5)).val)

4


### 3.1.4 [173. Binary Search Tree Iterator](https://leetcode.com/problems/binary-search-tree-iterator/) — ★★ · Medium

**Problem.** Iterator over a BST in ascending order with O(1) amortised `next` and O(h) memory.  
**Example.** tree `[7, 3, 15, null, null, 9, 20]`: `next()` → `3`, `next()` → `7`, `has_next()` → `True`, `next()` → `9`  
**Spot it.** Iterative in-order traversal, paused between calls.

**Ask first**
- Memory limit O(h)? → Yes → the stack, not a full list.

> 💡 **Idea.** A paused in-order walk: keep a stack of the left spine. `next()` pops a node and pushes the left spine of its right child.

**Brute force:** flatten into a sorted list up front — O(n) memory.

**Steps**
1. Constructor pushes the left spine of the root.
2. `next` pops a node (the next smallest).
3. Push the left spine of its right child.
4. `has_next` checks whether the stack is non-empty.

**Cost:** Time O(1) amortised per call (each node is pushed and popped once) · Space O(h)

**Likely follow-ups**
- **Why amortised O(1)?** Each node is pushed and popped once over the whole iteration.
- **Reverse iterator / `prev()` (LC 1586)?** Second stack for the right spine, or keep a history list of visited nodes with a pointer.
- **`peek()`?** Return `stack[-1].val` without popping.

In [34]:
class BSTIterator:
    """Lazy in-order iterator."""

    def __init__(self, root):
        self.stack = []
        self._push_left(root)

    def _push_left(self, node):
        # smallest unvisited keys sit along the left spine
        while node:
            self.stack.append(node)
            node = node.left

    def next(self):
        node = self.stack.pop()
        # the right subtree holds the next larger keys
        self._push_left(node.right)
        return node.val

    def has_next(self):
        return bool(self.stack)


it = BSTIterator(build_tree([7, 3, 15, None, None, 9, 20]))
out = []
while it.has_next():
    out.append(it.next())
assert out == [3, 7, 9, 15, 20]
print(out)

[3, 7, 9, 15, 20]


### 3.1.5 [285. Inorder Successor in BST](https://leetcode.com/problems/inorder-successor-in-bst/) — ★★ · Medium · Premium

**Problem.** The node with the smallest key greater than `p.val`.  
**Example.** `root = [2, 1, 3], p = 1` → the node `2`  
**Spot it.** BST descent remembering the last node where we went left.

**Ask first**
- Do nodes have parent pointers? → Not here. With parents it's LC 510 (see follow-ups).

> 💡 **Idea.** Walk down from the root. When `node.val > p.val`, it's a candidate — remember it and go left to find a smaller one. Otherwise go right.

**Brute force:** in-order list, find p, return the next — O(n).

**Steps**
1. Start at the root with no candidate.
2. `p.val < node.val` → record the node, go left.
3. Otherwise go right.
4. Return the last recorded candidate.

**Cost:** Time O(h) · Space O(1)

**Likely follow-ups**
- **Nodes have parent pointers but no root (LC 510)?** If p has a right child → leftmost of the right subtree. Else climb up until you come from a left child; that parent is the answer.
- **Predecessor?** Mirror: candidate when `node.val < p.val`, go right.
- **Plain binary tree (not BST)?** In-order traversal with a `prev` pointer; when `prev is p`, return the current node.

In [35]:
def inorder_successor(root, p):
    """Next larger node in a BST."""
    succ, node = None, root
    while node:
        # larger than p: a candidate; a smaller one may be to the left
        if p.val < node.val:
            succ, node = node, node.left
        # not larger: the successor is to the right
        else:
            node = node.right
    return succ


t = build_tree([5, 3, 6, 2, 4, None, None, 1])
assert inorder_successor(t, find(t, 4)).val == 5
assert inorder_successor(t, find(t, 6)) is None
print(inorder_successor(t, find(t, 1)).val)

2


### 3.1.6 [701. Insert into a Binary Search Tree](https://leetcode.com/problems/insert-into-a-binary-search-tree/) — ★★ · Medium

**Problem.** Insert a value into a BST (value not present); return the root.  
**Example.** `root = [4, 2, 7, 1, 3], val = 5` → `[4, 2, 7, 1, 3, 5]`  
**Spot it.** Walk down like a search; attach at the empty slot.

**Ask first**
- Must it stay balanced? → No, any valid BST is fine.

> 💡 **Idea.** Walk down like a search until you hit an empty spot; put the new node there.

**Brute force:** rebuild the whole tree from a sorted list — O(n).

**Steps**
1. Empty tree → new node.
2. Compare and choose left or right.
3. If that child is empty, place the new node there.
4. Otherwise continue from the child.

**Cost:** Time O(h) · Space O(1)

**Likely follow-ups**
- **Iterative?** Loop down, remember the parent, attach left or right. O(h) time, O(1) space.
- **Keep it balanced?** Self-balancing trees (AVL / red-black) rotate after inserts — explain the idea, nobody expects the code.
- **Duplicates?** Pick a side (e.g. right) or store a count in the node.

In [36]:
def insert_into_bst(root, val):
    """BST with val inserted."""
    if not root:
        return TreeNode(val)
    node = root
    while True:
        side = "left" if val < node.val else "right"
        child = getattr(node, side)
        # the search fell off here: this is the slot
        if not child:
            setattr(node, side, TreeNode(val))
            return root
        node = child


t = insert_into_bst(build_tree([4, 2, 7, 1, 3]), 5)
assert tree_to_list(t) == [4, 2, 7, 1, 3, 5]
print(tree_to_list(t))

[4, 2, 7, 1, 3, 5]


### 3.1.7 [450. Delete Node in a BST](https://leetcode.com/problems/delete-node-in-a-bst/) — ★★ · Medium

**Problem.** Delete the node with key `key` and return the root.  
**Example.** `root = [5, 3, 6, 2, 4, null, 7], key = 3` → `[5, 4, 6, 2, null, null, 7]`  
**Spot it.** Three cases: leaf, one child, two children (replace with in-order successor).

**Ask first**
- Must the result be a specific shape? → No, any valid BST.

> 💡 **Idea.** Find the node. 0 or 1 child: return the other child. 2 children: copy in the smallest value from the right subtree, then delete that value from the right subtree.

**Brute force:** in-order list without the key, rebuild a tree — O(n).

**Steps**
1. Recurse left or right to find the key.
2. Zero or one child → return that child (or `None`).
3. Two children → find the smallest node in the right subtree.
4. Copy its value up and delete it from the right subtree.

**Cost:** Time O(h) · Space O(h) recursion

**Likely follow-ups**
- **O(h) time?** The recursive search + successor trick above is O(h).
- **Without copying values (relink nodes)?** Detach the successor and put it in the deleted node's place, fixing parent links.
- **Delete a whole range (trim BST, LC 669)?** Recurse: if node < low return trim(right); if > high return trim(left); else trim both sides.

In [37]:
def delete_node(root, key):
    """BST with key removed."""
    if not root:
        return None
    if key < root.val:
        root.left = delete_node(root.left, key)
    elif key > root.val:
        root.right = delete_node(root.right, key)
    else:
        # at most one child: splice it in
        if not root.left or not root.right:
            return root.left or root.right
        # two children: borrow the in-order successor's value
        succ = root.right
        while succ.left:
            succ = succ.left
        root.val = succ.val
        # the successor has no left child, so this delete is easy
        root.right = delete_node(root.right, succ.val)
    return root


t = delete_node(build_tree([5, 3, 6, 2, 4, None, 7]), 3)
assert tree_to_list(t) == [5, 4, 6, 2, None, None, 7]
assert tree_to_list(delete_node(build_tree([5, 3, 6, 2, 4, None, 7]), 0)) == [
    5,
    3,
    6,
    2,
    4,
    None,
    7,
]
print(tree_to_list(t))

[5, 4, 6, 2, None, None, 7]


---
# 4) Trie

**The gist.** Store words letter by letter down paths from a root. Words that share a start share a path, so every node *is* a prefix.

> **Think of it like…** a dictionary's thumb tabs: open at "c", then "ca", then "cat" — each step narrows to words with that start, however big the book is.

Lookups cost O(word length), no matter how many words are stored. Here each node is a `dict` of children plus an end marker `"$"`.

**Spot it:** "starts with", "prefix", "autocomplete", "wildcard `.`", "many words searched on one grid".

**Classic bugs:** forgetting the end-of-word marker (`"app"` isn't a word just because `"apple"` is) · rebuilding the trie per query instead of once.

## 4.1 Prefix trees

Either build the trie once and walk it per query, or walk the trie *alongside* another search (wildcards, a grid DFS).

### 4.1.1 [208. Implement Trie (Prefix Tree)](https://leetcode.com/problems/implement-trie-prefix-tree/) — ★★★ · Medium

**Problem.** `insert`, `search` (whole word) and `starts_with` (prefix).  
**Example.** `insert("apple")`, `search("apple")` → `True`, `search("app")` → `False`, `starts_with("app")` → `True`  
**Spot it.** Nested dicts, one level per character, with an end-of-word marker.

**Ask first**
- Only lowercase letters? → Yes → 26-slot arrays or dicts both work; dicts are simpler and save memory on sparse tries.

> 💡 **Idea.** Nested dicts: each letter is a step down. `insert` creates missing steps and marks the end; `search` needs the end mark; `starts_with` just needs the path to exist.

**Brute force:** a set of words, scanning all of them for each prefix query — O(total length).

**Steps**
1. `insert`: walk/create one child per character, then set `"$"`.
2. `_walk(s)`: follow characters, returning `None` if a child is missing.
3. `search`: the walk ends at a node with `"$"`.
4. `starts_with`: the walk merely succeeds.

**Cost:** Time O(L) per operation · Space O(total characters inserted)

**Likely follow-ups**
- **Support delete, count words, count prefixes (LC 1804)?** Store `pass_count` and `end_count` per node; decrement on erase and drop children whose count hits 0.
- **Save memory?** Dict children instead of 26 arrays; compress single-child chains (radix tree).
- **Autocomplete top 3?** Cache the top suggestions in each node (see 1268).

In [38]:
class Trie:
    """Prefix tree over lowercase words."""

    def __init__(self):
        self.root = {}

    def insert(self, word):
        node = self.root
        # create the path one character at a time
        for ch in word:
            node = node.setdefault(ch, {})
        # mark that a whole word ends here
        node["$"] = True

    def _walk(self, s):
        node = self.root
        for ch in s:
            if ch not in node:
                return None
            node = node[ch]
        return node

    def search(self, word):
        # path exists and a word ends exactly here
        node = self._walk(word)
        return bool(node and node.get("$"))

    def starts_with(self, prefix):
        # any path at all is enough
        return self._walk(prefix) is not None


trie = Trie()
trie.insert("apple")
assert trie.search("apple") and not trie.search("app") and trie.starts_with("app")
trie.insert("app")
assert trie.search("app")
print("ok")

ok


### 4.1.2 [211. Design Add and Search Words Data Structure](https://leetcode.com/problems/design-add-and-search-words-data-structure/) — ★★★ · Medium

**Problem.** `add_word(word)`; `search(pattern)` where `.` matches any letter.  
**Example.** `add_word("bad")`, `add_word("dad")`, `search("pad")` → `False`, `search("b.d")` → `True`  
**Spot it.** Trie + DFS that branches on `.`.

**Ask first**
- How many dots per query? → At most 2 here, which keeps the branching small.

> 💡 **Idea.** Trie + DFS. A normal letter follows one child; a `.` tries every child.

**Brute force:** store words by length and compare each one letter by letter — fine for few words.

**Steps**
1. `add_word` is a normal trie insert.
2. `dfs(node, i)`: at the end of the pattern, check `"$"`.
3. `.` → try every child (skipping the `"$"` key).
4. A letter → follow that child if it exists.

**Cost:** Time O(L) to add; search is O(26^dots · L) in the worst case · Space O(total characters) + O(L) recursion

**Likely follow-ups**
- **Many dots make it slow?** Worst case O(26^dots · L). Bucket words by length to prune, or precompute patterns for short words.
- **Support `*` (any sequence)?** On `*`, try skipping it or consuming one letter and staying on `*` — becomes wildcard matching (LC 44) on a trie.
- **Need to delete words?** Add counts per node (like LC 1804).

In [39]:
class WordDictionary:
    """Trie supporting '.' wildcards."""

    def __init__(self):
        self.root = {}

    def add_word(self, word):
        node = self.root
        for ch in word:
            node = node.setdefault(ch, {})
        node["$"] = True

    def search(self, word):
        def dfs(node, i):
            # consumed the pattern: a word must end here
            if i == len(word):
                return "$" in node
            # wildcard: any child may continue the match
            if word[i] == ".":
                return any(
                    dfs(child, i + 1) for key, child in node.items() if key != "$"
                )
            # literal: exactly one edge to follow
            return word[i] in node and dfs(node[word[i]], i + 1)

        return dfs(self.root, 0)


wd = WordDictionary()
for w in ["bad", "dad", "mad"]:
    wd.add_word(w)
assert (
    not wd.search("pad") and wd.search("bad") and wd.search(".ad") and wd.search("b..")
)
print("ok")

ok


### 4.1.3 [212. Word Search II](https://leetcode.com/problems/word-search-ii/) — ★★★ · Hard

**Problem.** All dictionary words that can be traced in a letter grid (adjacent cells, no reuse).  
**Example.** `board = [["o","a","a","n"],["e","t","a","e"],["i","h","k","r"],["i","f","l","v"]], words = ["oath","pea","eat","rain"]` → `["eat", "oath"]`  
**Spot it.** Many words over one grid → trie of words + one DFS per cell.

**Ask first**
- Can a cell be reused in one word? → No → mark it while exploring, unmark after.

> 💡 **Idea.** Put all words in a trie; DFS the grid once, walking the trie alongside. When you reach a word's end, record it. Prune trie branches once their words are found.

**Brute force:** run Word Search (LC 79) separately for every word, O(words · cells · 4^L).

**Steps**
1. Insert all words into a trie, storing the whole word at its end node.
2. From each cell whose letter starts a trie edge, DFS: move into the child node.
3. If the node holds a word, record it and remove it (no duplicates).
4. Mark the cell visited, explore neighbours that continue the trie, restore it; prune empty trie nodes.

**Cost:** Time O(cells · 4 · 3^(L−1)) worst case; far less with pruning · Space O(total characters) for the trie + O(L) recursion

**Likely follow-ups**
- **Why prune found words?** Avoids re-finding them and shrinks the search a lot on big grids.

In [40]:
def find_words(board, words):
    """Dictionary words present in the grid."""
    # trie with the full word stored at its end node
    root = {}
    for w in words:
        node = root
        for ch in w:
            node = node.setdefault(ch, {})
        node["$"] = w
    rows, cols = len(board), len(board[0])
    found = []

    def dfs(r, c, parent):
        ch = board[r][c]
        node = parent[ch]
        # a complete word: record it once
        if "$" in node:
            found.append(node.pop("$"))
        # mark visited, explore neighbours that extend a trie path
        board[r][c] = "#"
        for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
            if 0 <= nr < rows and 0 <= nc < cols and board[nr][nc] in node:
                dfs(nr, nc, node)
        board[r][c] = ch
        # prune trie branches with nothing left to find
        if not node:
            parent.pop(ch)

    for r in range(rows):
        for c in range(cols):
            if board[r][c] in root:
                dfs(r, c, root)
    return found


board = [list("oaan"), list("etae"), list("ihkr"), list("iflv")]
assert sorted(find_words(board, ["oath", "pea", "eat", "rain"])) == ["eat", "oath"]
assert find_words([list("ab"), list("cd")], ["abcb"]) == []
print(find_words(board, ["oath", "pea", "eat", "rain"]))

['oath', 'eat']


### 4.1.4 [1268. Search Suggestions System](https://leetcode.com/problems/search-suggestions-system/) — ★ · Medium

**Problem.** After each typed character, return up to three lexicographically smallest products with that prefix.  
**Example.** `products = ["mobile","mouse","moneypot","monitor","mousepad"], search_word = "mouse"` → after `"m"`: `["mobile","moneypot","monitor"]`; after `"mou"`: `["mouse","mousepad"]`  
**Spot it.** Sorted list + binary search on the prefix (or a trie storing top-3 per node).

**Ask first**
- Suggestions in lexicographic order? → Yes, up to 3 per prefix.

> 💡 **Idea.** Sort the products. For each typed prefix, binary-search where it would start and take up to 3 matches from there.

**Brute force:** for every prefix, filter all products and sort — O(L · n log n).

**Steps**
1. Sort the products.
2. For each typed prefix, find the first position ≥ prefix with `bisect_left`.
3. Take up to three products from there that still start with the prefix.

**Cost:** Time O(n log n + L log n) (plus prefix checks) · Space O(n) for the sorted copy

**Likely follow-ups**
- **Trie version?** Each trie node stores up to 3 smallest words (insert products in sorted order). Each keystroke is O(1).
- **Rank by popularity instead?** Store a count per word; keep a size-3 top list (heap) at each node, update on insert.
- **Two pointers version?** Keep `[lo, hi]` over the sorted list and shrink it as each letter is typed.

In [41]:
def suggested_products(products, search_word):
    """Top-3 suggestions per typed prefix."""
    products = sorted(products)
    out, prefix = [], ""
    for ch in search_word:
        prefix += ch
        # words with this prefix start here, in sorted order
        i = bisect_left(products, prefix)
        out.append([p for p in products[i : i + 3] if p.startswith(prefix)])
    return out


res = suggested_products(
    ["mobile", "mouse", "moneypot", "monitor", "mousepad"], "mouse"
)
assert res[0] == ["mobile", "moneypot", "monitor"]
assert res[-1] == ["mouse", "mousepad"]
print(res)

[['mobile', 'moneypot', 'monitor'], ['mobile', 'moneypot', 'monitor'], ['mouse', 'mousepad'], ['mouse', 'mousepad'], ['mouse', 'mousepad']]


---
# 5) Heap / Priority Queue

**The gist.** A heap always hands you the smallest item, at O(log n) per push or pop, without sorting everything.

> **Think of it like…** hospital triage: patients arrive in any order, but the most urgent is always seen next. Nobody sorts the waiting room.

Python's `heapq` is a **min-heap**; push `-x` for a max-heap; tuples compare element by element, so `(priority, tie_breaker, item)` works.

| Goal | Heap | Size |
|---|---|---|
| k largest | **min**-heap, pop when size > k | k |
| k smallest | **max**-heap (push negatives) | k |
| keep taking the best | heap of candidates | all |
| median of a stream | max-heap (low half) + min-heap (high half) | n |
| merge k sorted sources | min-heap of current heads | k |

**Spot it:** "k largest / smallest / closest / most frequent", "stream", "schedule", "merge k sorted", "median".

**Classic bugs:** using a max-heap for top-k largest (you want to evict the *smallest* kept item) · comparing non-comparable objects in tuples (add an index tie-breaker).

## 5.1 Top-k and k-th element

For "top k", keep a heap of size k whose top is the *worst* item you're keeping — one comparison decides whether a newcomer gets in.

### 5.1.1 [703. Kth Largest Element in a Stream](https://leetcode.com/problems/kth-largest-element-in-a-stream/) — ★★ · Easy

**Problem.** After each `add`, return the k-th largest element so far.  
**Example.** `k = 3, nums = [4, 5, 8, 2]`: `add(3)` → `4`, `add(5)` → `5`, `add(10)` → `5`  
**Spot it.** k-th largest → min-heap of size k.

**Ask first**
- Fewer than k numbers at the start? → Possible — still push until size k.

> 💡 **Idea.** Keep a min-heap of the k largest so far. Its top is the k-th largest.

**Brute force:** sort on every add, O(n log n) per call.

**Steps**
1. Push the new value.
2. If the heap has more than k items, pop the smallest.
3. Return the top.

**Cost:** Time O(log k) per add · Space O(k)

In [42]:
class KthLargest:
    """Running k-th largest value."""

    def __init__(self, k, nums):
        self.k = k
        # min-heap of the k largest values seen
        self.heap = []
        for x in nums:
            self.add(x)

    def add(self, val):
        heapq.heappush(self.heap, val)
        # drop the smallest so only the top k remain
        if len(self.heap) > self.k:
            heapq.heappop(self.heap)
        # smallest of the top k = k-th largest
        return self.heap[0]


kl = KthLargest(3, [4, 5, 8, 2])
assert [kl.add(v) for v in [3, 5, 10, 9, 4]] == [4, 5, 5, 8, 8]
print("ok")

ok


### 5.1.2 [1046. Last Stone Weight](https://leetcode.com/problems/last-stone-weight/) — ★★ · Easy

**Problem.** Repeatedly smash the two heaviest stones; return the last weight (or 0).  
**Example.** `[2, 7, 4, 1, 8, 1]` → `1`  
**Spot it.** Repeatedly take the two largest → max-heap.

**Ask first**
- Return 0 if no stones left? → Yes.

> 💡 **Idea.** Max-heap (push negatives). Pop the two heaviest, push back the difference if non-zero.

**Brute force:** sort the list after every smash, O(n² log n).

**Steps**
1. Heapify the negated weights.
2. Pop the two heaviest.
3. If they differ, push back the difference.
4. Return the last stone or 0.

**Cost:** Time O(n log n) · Space O(n)

In [43]:
def last_stone_weight(stones):
    """Weight of the final stone, or 0."""
    # negate for a max-heap
    heap = [-s for s in stones]
    heapq.heapify(heap)
    while len(heap) > 1:
        # two heaviest stones
        a, b = -heapq.heappop(heap), -heapq.heappop(heap)
        if a != b:
            heapq.heappush(heap, -(a - b))
    return -heap[0] if heap else 0


assert last_stone_weight([2, 7, 4, 1, 8, 1]) == 1
assert last_stone_weight([1]) == 1
print(last_stone_weight([2, 7, 4, 1, 8, 1]))

1


### 5.1.3 [973. K Closest Points to Origin](https://leetcode.com/problems/k-closest-points-to-origin/) — ★★ · Medium

**Problem.** The `k` points closest to the origin.  
**Example.** `points = [[1, 3], [-2, 2]], k = 1` → `[[-2, 2]]`  
**Spot it.** k smallest by distance → max-heap of size k (or `heapq.nsmallest`).

**Ask first**
- Any order for the answer? → Yes.
- Compare squared distances? → Yes, skip `sqrt`.

> 💡 **Idea.** Max-heap of size k keyed by squared distance: push each point, pop the farthest when the heap gets bigger than k.

**Brute force:** sort all points by distance, O(n log n).

**Steps**
1. Push `(−distance², point)`.
2. If the heap exceeds k, pop (removes the farthest).
3. Return the points left.

**Cost:** Time O(n log k) · Space O(k)

**Likely follow-ups**
- **Points arrive as a stream?** The size-k max-heap works as-is, O(k) memory. Quickselect needs all points up front.
- **Faster on average?** Quickselect on distances: O(n) average, O(n²) worst — use a random pivot.
- **Too many points for memory?** Split into chunks, take each chunk's k closest, merge those.
- **Closest to an arbitrary point / repeated queries?** Precompute a k-d tree for nearest-neighbour queries.

In [44]:
def k_closest(points, k):
    """k points nearest the origin."""
    # max-heap on distance (negated); squared distance avoids sqrt
    heap = []
    for x, y in points:
        heapq.heappush(heap, (-(x * x + y * y), [x, y]))
        # drop the farthest of the kept points
        if len(heap) > k:
            heapq.heappop(heap)
    return [p for _, p in heap]


assert k_closest([[1, 3], [-2, 2]], 1) == [[-2, 2]]
assert sorted(k_closest([[3, 3], [5, -1], [-2, 4]], 2)) == [[-2, 4], [3, 3]]
print(k_closest([[3, 3], [5, -1], [-2, 4]], 2))

[[-2, 4], [3, 3]]


### 5.1.4 [215. Kth Largest Element in an Array](https://leetcode.com/problems/kth-largest-element-in-an-array/) — ★★ · Medium

**Problem.** k-th largest element (not distinct), ideally better than sorting.  
**Example.** `[3, 2, 1, 5, 6, 4], k = 2` → `5`  
**Spot it.** Min-heap of size k (O(n log k)) or Quickselect (O(n) average).

**Ask first**
- Can I modify the array? → Quickselect does; the heap doesn't.

> 💡 **Idea.** Quickselect: split around a random pivot into bigger / equal / smaller, then recurse only into the part that holds the k-th largest. (A size-k min-heap also works: O(n log k).)

**Brute force:** sort and index, O(n log n).

**Steps**
1. Pick a random pivot.
2. Split into greater / equal / less.
3. If k falls in "greater", recurse there; if in "equal", return the pivot.
4. Otherwise recurse into "less" with k reduced.

**Cost:** Time O(n) average (each step keeps a fraction of the list), O(n²) worst · Space O(n) for the lists here (in-place partitioning gets O(1))

**Likely follow-ups**
- **Quickselect worst case?** O(n²) with bad pivots — random pivot makes it O(n) expected; median-of-medians guarantees O(n).
- **Stream of numbers?** Size-k min-heap (that's LC 703).
- **Lots of duplicates?** 3-way partition so equal values don't slow quickselect down.
- **Values in a small range?** Counting sort, O(n + range).

In [45]:
def find_kth_largest(nums, k):
    """k-th largest value via quickselect."""
    pivot = random.choice(nums)
    # three-way split handles duplicates
    greater = [x for x in nums if x > pivot]
    equal = [x for x in nums if x == pivot]
    # answer among the larger values
    if k <= len(greater):
        return find_kth_largest(greater, k)
    # answer is the pivot itself
    if k <= len(greater) + len(equal):
        return pivot
    # answer among the smaller values, with rank adjusted
    less = [x for x in nums if x < pivot]
    return find_kth_largest(less, k - len(greater) - len(equal))


assert find_kth_largest([3, 2, 1, 5, 6, 4], 2) == 5
assert find_kth_largest([3, 2, 3, 1, 2, 4, 5, 5, 6], 4) == 4
print(find_kth_largest([3, 2, 1, 5, 6, 4], 2))

5


### 5.1.5 [692. Top K Frequent Words](https://leetcode.com/problems/top-k-frequent-words/) — ★★ · Medium

**Problem.** `k` most frequent words; ties broken alphabetically.  
**Example.** `["i","love","leetcode","i","love","coding"], k = 2` → `["i", "love"]`  
**Spot it.** Top k with a composite key `(−count, word)`.

**Ask first**
- Tie-break? → Alphabetical — this is what makes the heap key tricky.

> 💡 **Idea.** Count words, then pick the top k with the order `(-count, word)` — higher count first, alphabetical on ties.

**Brute force:** sort all distinct words by `(-count, word)`, O(m log m).

**Steps**
1. Count words.
2. `heapq.nsmallest(k, words, key=(−count, word))`.

**Cost:** Time O(n log k) · Space O(n) for the counts

**Likely follow-ups**
- **O(n log k)?** Size-k min-heap whose *worst* item is on top: smaller count, or same count but alphabetically *later*. Python: wrap words in a class with a reversed `<`.
- **Doesn't fit in memory / stream?** Shard by hash of the word, count each shard, take each shard's top k, merge. Approximate: Count-Min Sketch + heap.
- **Bucket sort version?** Bucket by count, sort words inside a bucket — O(n + m log m) worst case.

In [46]:
def top_k_frequent_words(words, k):
    """k most frequent words, alphabetical on ties."""
    count = Counter(words)
    # higher count first, then alphabetical
    return heapq.nsmallest(k, count, key=lambda w: (-count[w], w))


assert top_k_frequent_words(["i", "love", "leetcode", "i", "love", "coding"], 2) == [
    "i",
    "love",
]
words = ["the", "day", "is", "sunny", "the", "the", "the", "sunny", "is", "is"]
assert top_k_frequent_words(words, 4) == ["the", "is", "sunny", "day"]
print(top_k_frequent_words(words, 4))

['the', 'is', 'sunny', 'day']


### 5.1.6 [373. Find K Pairs with Smallest Sums](https://leetcode.com/problems/find-k-pairs-with-smallest-sums/) — ★★ · Medium

**Problem.** Sorted arrays `a`, `b`; the `k` pairs `(a[i], b[j])` with the smallest sums.  
**Example.** `a = [1, 7, 11], b = [2, 4, 6], k = 3` → `[[1, 2], [1, 4], [1, 6]]`  
**Spot it.** Merge k sorted lists: row `i` is `a[i] + b[0], a[i] + b[1], …`.

**Ask first**
- Are both arrays sorted? → Yes, that's what lets you push just one neighbour.

> 💡 **Idea.** Like merging k sorted lists: start with `(nums1[i] + nums2[0])` for the first k values of nums1. Pop the smallest pair, push the same `i` with the next `j`.

**Brute force:** build all pairs and sort, O(m·n log(m·n)).

**Steps**
1. Seed the heap with `(a[i] + b[0], i, 0)` for the first k values of `i`.
2. Pop the smallest; record the pair.
3. Push `(a[i] + b[j + 1], i, j + 1)` if it exists.
4. Stop after k pairs.

**Cost:** Time O(k log k) · Space O(k)

**Likely follow-ups**
- **k-th smallest *sum* only?** Binary search on the value and count pairs with sum ≤ mid using two pointers: O((m + n) log range).
- **Same idea in a matrix (LC 378)?** Rows are the `i`s, columns the `j`s — identical heap.
- **k-th smallest among m sorted rows picking one per row (LC 1439)?** Merge rows two at a time keeping only the k smallest sums.

In [47]:
def k_smallest_pairs(a, b, k):
    """k pairs with the smallest sums."""
    # head of each row i: pair (i, 0)
    heap = [(a[i] + b[0], i, 0) for i in range(min(k, len(a)))]
    out = []
    while heap and len(out) < k:
        _, i, j = heapq.heappop(heap)
        out.append([a[i], b[j]])
        # next pair in the same row
        if j + 1 < len(b):
            heapq.heappush(heap, (a[i] + b[j + 1], i, j + 1))
    return out


assert k_smallest_pairs([1, 7, 11], [2, 4, 6], 3) == [[1, 2], [1, 4], [1, 6]]
assert k_smallest_pairs([1, 1, 2], [1, 2, 3], 2) == [[1, 1], [1, 1]]
print(k_smallest_pairs([1, 7, 11], [2, 4, 6], 3))

[[1, 2], [1, 4], [1, 6]]


## 5.2 Scheduling and simulation with heaps

Things become *available* over time and you always act on the best available one. Sort by availability, push items as they become available, pop the best.

### 5.2.1 [621. Task Scheduler](https://leetcode.com/problems/task-scheduler/) — ★★ · Medium

**Problem.** Tasks with a cooldown `n` between equal tasks; minimum time slots (idle allowed).  
**Example.** `tasks = ["A","A","A","B","B","B"], n = 2` → `8` (A B idle A B idle A B)  
**Spot it.** The most frequent task dictates the frame → counting formula.

**Ask first**
- Can tasks run in any order? → Yes (if order is fixed, it's LC 2365).

> 💡 **Idea.** The most frequent task sets the frame: `(max_count − 1)` gaps of size `n + 1`, plus one slot for each task tied for the max. If there are more tasks than that, no idling is needed: answer = `len(tasks)`.

**Brute force:** simulate time with a max-heap and a cooldown queue — O(total time · log 26), still fine.

**Steps**
1. Count tasks; `max_count` = highest count; `ties` = tasks with that count.
2. Frame length = `(max_count − 1) × (n + 1) + ties`.
3. Answer = `max(len(tasks), frame)`.

**Cost:** Time O(N) · Space O(26)

**Likely follow-ups**
- **Return the actual schedule?** Simulate with a max-heap of counts plus a queue of `(ready_time, task)`; output idle when the heap is empty.
- **Tasks must stay in the given order (LC 2365)?** Track `last_day[task]`; each task runs at `max(day + 1, last_day + n + 1)`.
- **Same frequency — deterministic tie order?** Add a tie-breaker (task name) to the heap key.

In [48]:
def least_interval(tasks, n):
    """Fewest slots to run all tasks with cooldown n."""
    counts = Counter(tasks).values()
    max_count = max(counts)
    # tasks that also need a slot in the final row
    ties = sum(1 for c in counts if c == max_count)
    # frame forced by the busiest task, or no idle time at all
    return max(len(tasks), (max_count - 1) * (n + 1) + ties)


assert least_interval(list("AAABBB"), 2) == 8
assert least_interval(list("ACABDB"), 1) == 6
assert least_interval(list("AAABBB"), 3) == 10
print(least_interval(list("AAABBB"), 2))

8


### 5.2.2 [767. Reorganize String](https://leetcode.com/problems/reorganize-string/) — ★★ · Medium

**Problem.** Rearrange so no two adjacent characters are equal, or return `""`.  
**Example.** `"aab"` → `"aba"`; `"aaab"` → `""`  
**Spot it.** Always place the most frequent remaining character that isn't the previous one → max-heap.

**Ask first**
- Return any valid answer? → Yes, or `""` if impossible.

> 💡 **Idea.** Always place the most common remaining letter that isn't the one you just placed (max-heap, hold the last one back for one turn). Impossible if any letter appears more than `(n + 1) // 2` times.

**Brute force:** try permutations, factorial.

**Steps**
1. Max-heap of `(−count, char)`.
2. Pop the top and append it.
3. Push back the previously held character (if it still has copies).
4. Hold the current one; if the output is shorter than `s` at the end, return `""`.

**Cost:** Time O(n log 26) · Space O(26)

**Likely follow-ups**
- **Same letters at least k apart (LC 358)?** Max-heap + a waiting queue of length k: a used letter comes back to the heap after k steps.
- **O(n) without a heap?** Put the most frequent letter on even indices 0, 2, 4…, then fill the rest of the even and then odd slots.
- **Count arrangements?** That's combinatorics / DP — rarely asked; mention inclusion–exclusion.

In [49]:
def reorganize_string(s):
    """Arrangement with no equal neighbours, or empty."""
    heap = [(-c, ch) for ch, c in Counter(s).items()]
    heapq.heapify(heap)
    out = []
    prev = None
    while heap:
        # most frequent character that is allowed now
        c, ch = heapq.heappop(heap)
        out.append(ch)
        # the character held last round may be used again
        if prev:
            heapq.heappush(heap, prev)
        # hold this one out for a round (if copies remain)
        prev = (c + 1, ch) if c + 1 < 0 else None
    # leftover held copies mean it was impossible
    return "".join(out) if len(out) == len(s) else ""


res = reorganize_string("aab")
assert res == "aba"
assert reorganize_string("aaab") == ""
assert all(
    a != b for a, b in zip(reorganize_string("aaabbc"), reorganize_string("aaabbc")[1:])
)
print(res)

aba


### 5.2.3 [355. Design Twitter](https://leetcode.com/problems/design-twitter/) — ★★ · Medium

**Problem.** `post_tweet`, `follow`, `unfollow`, and `get_news_feed` (10 most recent tweets from the user and followees).  
**Example.** user 1 `post_tweet(5)`, `get_news_feed(1)` → `[5]`, user 1 `follow(2)`, user 2 `post_tweet(6)`, `get_news_feed(1)` → `[6, 5]`  
**Spot it.** Merge k sorted tweet lists → heap over each followee's latest tweet.

**Ask first**
- Can a user follow themselves? → Ignore it (they always see their own tweets).

> 💡 **Idea.** Each user keeps a list of `(time, tweet)`. The feed merges the latest tweets of the user and everyone they follow with a heap and stops at 10.

**Brute force:** collect all tweets of all followees, sort, take 10 — O(total tweets log).

**Steps**
1. Store `(time, tweet_id)` per user; time decreases so smaller = newer.
2. Feed: seed a heap with each followee's (and own) newest tweet.
3. Pop up to 10 times, pushing that user's next older tweet.
4. Follow / unfollow update a set (no self-follow).

**Cost:** Time O(F + 10 log F) per feed, O(1) post/follow · Space O(total tweets + follows)

**Likely follow-ups**
- **Millions of users — design it for real?** Fan-out on write: push each tweet into followers' cached timelines (fast reads). Celebrities: fan-out on read and merge at read time (hybrid).
- **Paging ("next 10")?** Pass a cursor (last timestamp); the heap merge resumes from there.
- **Unfollow mid-merge or deleted tweets?** Filter while merging; for deleted tweets keep a tombstone set.

In [50]:
class Twitter:
    """Minimal Twitter feed."""

    def __init__(self):
        # decreasing counter: newer tweets compare smaller
        self.time = 0
        self.tweets = defaultdict(list)
        self.following = defaultdict(set)

    def post_tweet(self, user_id, tweet_id):
        self.time -= 1
        self.tweets[user_id].append((self.time, tweet_id))

    def get_news_feed(self, user_id):
        # newest tweet of each followed user (and the user)
        heap = []
        for u in self.following[user_id] | {user_id}:
            if self.tweets[u]:
                i = len(self.tweets[u]) - 1
                heap.append((self.tweets[u][i][0], u, i))
        heapq.heapify(heap)
        feed = []
        while heap and len(feed) < 10:
            _, u, i = heapq.heappop(heap)
            feed.append(self.tweets[u][i][1])
            # that user's next older tweet joins the merge
            if i:
                heapq.heappush(heap, (self.tweets[u][i - 1][0], u, i - 1))
        return feed

    def follow(self, follower, followee):
        if follower != followee:
            self.following[follower].add(followee)

    def unfollow(self, follower, followee):
        self.following[follower].discard(followee)


tw = Twitter()
tw.post_tweet(1, 5)
assert tw.get_news_feed(1) == [5]
tw.follow(1, 2)
tw.post_tweet(2, 6)
assert tw.get_news_feed(1) == [6, 5]
tw.unfollow(1, 2)
assert tw.get_news_feed(1) == [5]
print("ok")

ok


### 5.2.4 [295. Find Median from Data Stream](https://leetcode.com/problems/find-median-from-data-stream/) — ★★★ · Hard

**Problem.** `add_num` and `find_median` over a stream.  
**Example.** `add_num(1)`, `add_num(2)`, `find_median()` → `1.5`, `add_num(3)`, `find_median()` → `2.0`  
**Spot it.** Two heaps: max-heap for the lower half, min-heap for the upper half.

**Ask first**
- Even count → average of the two middles? → Yes.

> 💡 **Idea.** Two heaps: a max-heap for the lower half, a min-heap for the upper half, kept within one in size. The median is on top.

**Brute force:** keep a sorted list and insert with bisect — O(n) per add.

**Steps**
1. Push the new number into the lower (max) heap.
2. Move the lower heap's largest into the upper heap (keeps every low ≤ every high).
3. If the upper heap is now bigger, move its smallest back.
4. Median = top of lower (odd count) or the average of both tops.

**Cost:** Time O(log n) per add, O(1) per median · Space O(n)

**Likely follow-ups**
- **All numbers are in 0..100?** Count array of 101 buckets; walk it to the middle — O(100) per query.
- **99% of numbers are in 0..100?** Buckets for 0..100 plus two counters (or heaps) for outliers below and above.
- **Sliding-window median (LC 480)?** Two heaps with lazy deletion, or a sorted list.

In [51]:
class MedianFinder:
    """Streaming median with two heaps."""

    def __init__(self):
        # low: max-heap (negated) of the smaller half; high: min-heap of the larger
        self.low = []
        self.high = []

    def add_num(self, num):
        heapq.heappush(self.low, -num)
        # move the largest low value up to keep the halves ordered
        heapq.heappush(self.high, -heapq.heappop(self.low))
        # rebalance: low keeps the extra element
        if len(self.high) > len(self.low):
            heapq.heappush(self.low, -heapq.heappop(self.high))

    def find_median(self):
        if len(self.low) > len(self.high):
            return float(-self.low[0])
        return (-self.low[0] + self.high[0]) / 2


mf = MedianFinder()
mf.add_num(1)
mf.add_num(2)
assert mf.find_median() == 1.5
mf.add_num(3)
assert mf.find_median() == 2.0
print(mf.find_median())

2.0


### 5.2.5 [1834. Single-Threaded CPU](https://leetcode.com/problems/single-threaded-cpu/) — ★★ · Medium

**Problem.** Tasks `[enqueue_time, processing_time]`; the CPU always picks the available task with the shortest processing time (ties by index). Return the processing order.  
**Example.** `tasks = [[1,2],[2,4],[3,2],[4,1]]` → `[0, 2, 3, 1]`  
**Spot it.** Event simulation: sort by enqueue time + min-heap of available tasks.

**Ask first**
- Tie-break? → Shorter processing time, then smaller index.

> 💡 **Idea.** Sort tasks by enqueue time. Keep a min-heap of available tasks keyed by `(processing_time, index)`. When idle with an empty heap, jump time to the next enqueue time.

**Brute force:** at each step scan all tasks for the best available one, O(n²).

**Steps**
1. Sort task indices by enqueue time.
2. If nothing is available, jump the clock to the next enqueue time.
3. Push every task enqueued by now onto the heap.
4. Pop and run the best; advance the clock by its duration.

**Cost:** Time O(n log n) · Space O(n)

**Likely follow-ups**
- **k CPUs instead of one (LC 1882)?** Second min-heap of servers keyed by `(free_time, id)`; pop a server, give it the best available task, push it back with its new free time.
- **Priority instead of shortest-job-first?** Change the heap key; for fairness, add aging (bump waiting tasks' priority over time).
- **Tasks can be cancelled?** Lazy deletion: mark cancelled ids and skip them when popped.

In [52]:
def get_order(tasks):
    """Indices in the order the CPU processes them."""
    order = sorted(range(len(tasks)), key=lambda i: tasks[i][0])
    # available tasks keyed by (processing time, index)
    heap = []
    out, time, i = [], 0, 0
    while len(out) < len(tasks):
        # idle CPU: jump to the next arrival
        if not heap and time < tasks[order[i]][0]:
            time = tasks[order[i]][0]
        # everything that has arrived joins the queue
        while i < len(order) and tasks[order[i]][0] <= time:
            heapq.heappush(heap, (tasks[order[i]][1], order[i]))
            i += 1
        # run the shortest available task to completion
        duration, idx = heapq.heappop(heap)
        time += duration
        out.append(idx)
    return out


assert get_order([[1, 2], [2, 4], [3, 2], [4, 1]]) == [0, 2, 3, 1]
assert get_order([[7, 10], [7, 12], [7, 5], [7, 4], [7, 2]]) == [4, 3, 2, 0, 1]
print(get_order([[1, 2], [2, 4], [3, 2], [4, 1]]))

[0, 2, 3, 1]


### 5.2.6 [2542. Maximum Subsequence Score](https://leetcode.com/problems/maximum-subsequence-score/) — ★★ · Medium

**Problem.** Pick `k` indices; the score is (sum of their `nums1`) × (minimum of their `nums2`). Return the best score.  
**Example.** `nums1 = [1, 3, 3, 2], nums2 = [2, 1, 3, 4], k = 3` → `12` (indices 0, 2, 3: (1 + 3 + 2) × min(2, 3, 4) = 12)  
**Spot it.** "Sum of k" times "min of k" → fix the minimum by sorting on `nums2`, keep the best sum with a size-k heap.

**Ask first**
- Pick exactly k indices? → Yes, any k (not contiguous).

> 💡 **Idea.** Sort pairs by `nums2` descending. Walking down, the current `nums2` is the minimum of everything chosen so far. Keep the k largest `nums1` values in a min-heap with their sum; score = sum × current `nums2`.

**Brute force:** try every subset of size k, C(n, k).

**Steps**
1. Sort pairs `(nums2, nums1)` by `nums2`, largest first.
2. Push each `nums1` onto a min-heap and add it to a running sum; past `k` values, pop the smallest.
3. With exactly `k` values in the heap, score = sum × current `nums2`; keep the best.

**Cost:** Time O(n log n) (the sort, plus one push and at most one pop per pair) · Space O(n) (the sorted pairs (the heap holds at most k + 1 values))

**Likely follow-ups**
- **Why sort by `nums2`?** Fixing each element as the minimum turns the problem into "best k sum among elements with `nums2` ≥ this" — a heap handles that.
- **Same trick elsewhere?** Max performance of a team (LC 1383), hire k workers (LC 857).

In [53]:
def max_subsequence_score(nums1, nums2, k):
    """Best (sum of k nums1 values) × (min of their nums2 values)."""
    pairs = sorted(zip(nums2, nums1), reverse=True)
    heap, total, best = [], 0, 0
    for low, value in pairs:
        # every earlier pair has nums2 >= low, so low is the minimum here
        heapq.heappush(heap, value)
        total += value
        # keep only the k largest nums1 values seen so far
        if len(heap) > k:
            total -= heapq.heappop(heap)
        if len(heap) == k:
            best = max(best, total * low)
    return best


assert max_subsequence_score([1, 3, 3, 2], [2, 1, 3, 4], 3) == 12
assert max_subsequence_score([4, 2, 3, 1, 1], [7, 5, 10, 9, 6], 1) == 30
print(max_subsequence_score([1, 3, 3, 2], [2, 1, 3, 4], 3))

12


### 5.2.7 [1642. Furthest Building You Can Reach](https://leetcode.com/problems/furthest-building-you-can-reach/) — ★ · Medium

**Problem.** Climbing up needs bricks (height difference) or one ladder. How far can you go?  
**Example.** `heights = [4, 2, 7, 6, 9, 14, 12], bricks = 5, ladders = 1` → `4`  
**Spot it.** Use ladders on the largest climbs → min-heap of the climbs currently given ladders.

**Ask first**
- Ladders cover any height? → Yes, one climb each.

> 💡 **Idea.** Use ladders on the biggest climbs. Walk forward pushing each climb onto a min-heap; when you have more climbs than ladders, pay the smallest one with bricks. Stop when bricks run out.

**Brute force:** try all ways of assigning ladders, exponential.

**Steps**
1. For each upward climb, push it onto a min-heap (tentative ladder).
2. If the heap holds more climbs than ladders, pop the smallest and pay it in bricks.
3. If bricks go negative, the current building is the last reachable.
4. Otherwise you reach the end.

**Cost:** Time O(n log L) (heap size ≤ ladders + 1) · Space O(L)

**Likely follow-ups**
- **Binary-search version?** Can I reach building i? Sort the first i climbs; ladders take the biggest, bricks the rest. Monotone in i → binary search, O(n log² n).
- **Minimum bricks to reach the end with L ladders?** Same heap: total of the climbs not covered by the L biggest.

In [54]:
def furthest_building(heights, bricks, ladders):
    """Furthest reachable index."""
    # climbs currently covered by ladders (smallest on top)
    heap = []
    for i in range(len(heights) - 1):
        climb = heights[i + 1] - heights[i]
        if climb <= 0:
            continue
        heapq.heappush(heap, climb)
        # out of ladders: the smallest laddered climb is paid in bricks
        if len(heap) > ladders:
            bricks -= heapq.heappop(heap)
            if bricks < 0:
                return i
    return len(heights) - 1


assert furthest_building([4, 2, 7, 6, 9, 14, 12], 5, 1) == 4
assert furthest_building([4, 12, 2, 7, 3, 18, 20, 3, 19], 10, 2) == 7
assert furthest_building([14, 3, 19, 3], 17, 0) == 3
print(furthest_building([4, 2, 7, 6, 9, 14, 12], 5, 1))

4


---
# Summary

| Pattern | Recognise it by | Core move | Cost |
|---|---|---|---|
| Bottom-up DFS | height, diameter, balanced, max path, LCA | combine child results, update global best | O(n) |
| Tree DP (state pair) | rob/skip | return a tuple of states per node | O(n) |
| Top-down DFS | path sums, good nodes, bounds | pass state down, backtrack shared lists | O(n) |
| Prefix sum on paths | downward paths summing to k | hash map of prefix sums, undo on return | O(n) |
| Construction | pre/in, in/post, serialise | root from one order, split by the other | O(n) |
| BFS by level | level lists, right view, width | `for _ in range(len(queue))` | O(n) |
| BST bounds / in-order | validate, k-th, successor, recover | `(low, high)` window; in-order is sorted | O(h)–O(n) |
| Trie | prefix queries, many words on one grid | nested dicts + end marker | O(L) |
| Heap size k | top-k, k-th, k closest | min-heap for largest, max-heap for smallest | O(n log k) |
| Two heaps | streaming median | lower max-heap + upper min-heap | O(log n) |
| k-way merge | merge lists, smallest pairs, feed | heap of current heads | O(N log k) |
| Greedy + heap | ladders, CPU scheduling, reorganise | sort by availability, heap by value | O(n log n) |